In [ ]:
import sys
from pathlib import Path

print("Python executable:")
print(sys.executable)

print("\nCurrent directory:")
print(Path.cwd())

STEP27_H5AD = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step27_Trajectory/"
    "Data/GSE149614_STEP27_FINAL_TRAJECTORY.h5ad"
)

print("\nStep27 input:")
print(STEP27_H5AD)

print("\nExists:")
print(STEP27_H5AD.exists())

In [ ]:
# =============================================================================
# Step29A Cell 2
# Environment audit
# =============================================================================

import sys
import os
import json
import platform
import importlib.util

from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp

import scanpy as sc
import anndata as ad

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())

print("\nScanpy:", sc.__version__)
print("AnnData:", ad.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step29A Cell 3
# Step29 project directories
# =============================================================================

ROOT = Path(_hcc_path(''))

STEP29_DIR = (
    ROOT /
    "LIHC_scRNA_Step29_GRN"
)

DATA_DIR = STEP29_DIR / "Data"
TABLE_DIR = STEP29_DIR / "Tables"
FIG_DIR = STEP29_DIR / "Figures"
AUDIT_DIR = STEP29_DIR / "Audit"

for folder in [
    STEP29_DIR,
    DATA_DIR,
    TABLE_DIR,
    FIG_DIR,
    AUDIT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("STEP29_DIR:", STEP29_DIR)
print("Exists:", STEP29_DIR.exists())

In [ ]:
# =============================================================================
# Step29A Cell 4
# Frozen Step27 input lock
# =============================================================================

STEP27_H5AD = (
    ROOT /
    "LIHC_scRNA_Step27_Trajectory" /
    "Data" /
    "GSE149614_STEP27_FINAL_TRAJECTORY.h5ad"
)

print("Input:")
print(STEP27_H5AD)

print("\nExists:")
print(STEP27_H5AD.exists())

assert STEP27_H5AD.exists()

print("\nSTEP27 INPUT LOCK: PASS")

In [ ]:
# =============================================================================
# Step29A Cell 5
# Load frozen malignant-cell object
# =============================================================================

adata29 = sc.read_h5ad(
    STEP27_H5AD
)

print(
    "Shape:",
    adata29.shape
)

assert adata29.n_obs == 12057
assert adata29.n_vars == 25712

print(
    "Frozen malignant universe: PASS"
)

In [ ]:
# =============================================================================
# Step29A Cell 6
# Frozen annotation audit
# =============================================================================

STATE_COL = "final_malignant_state_id"
PATIENT_COL = "patient"
PT_COL = "MS6_anchored_pseudotime"

required_obs = [
    STATE_COL,
    PATIENT_COL,
    PT_COL,
    "MS3_conserved_Core50_score",
    "Program__Proliferation_G2M",
    "DeepProgram__Mitotic_Entry",
    "DeepProgram__Spindle_Chromosome_Segregation",
    "DeepProgram__Hepatic_Differentiation",
    "DeepProgram__Bile_Acid_Metabolism"
]

missing_obs = [
    col
    for col in required_obs
    if col not in adata29.obs.columns
]

print(
    "Missing required fields:",
    missing_obs
)

assert len(missing_obs) == 0

print(
    "Frozen annotations: PASS"
)

In [ ]:
# =============================================================================
# Step29A Cell 7
# Frozen state and patient census
# =============================================================================

state_counts = (
    adata29.obs[
        STATE_COL
    ]
    .astype(str)
    .value_counts()
    .sort_index()
)

patient_counts = (
    adata29.obs[
        PATIENT_COL
    ]
    .astype(str)
    .value_counts()
    .sort_index()
)

patient_state = pd.crosstab(
    adata29.obs[
        PATIENT_COL
    ].astype(str),

    adata29.obs[
        STATE_COL
    ].astype(str)
)

print("State counts:")
display(
    state_counts.to_frame("N")
)

print("\nPatient counts:")
display(
    patient_counts.to_frame("N")
)

print("\nPatient × State:")
display(
    patient_state
)


In [ ]:
# =============================================================================
# Step29A Cell 8
# Expression matrix audit
# =============================================================================

X = adata29.X

print(
    "X type:",
    type(X)
)

print(
    "Sparse:",
    sp.issparse(X)
)

print(
    "Shape:",
    X.shape
)

print(
    "Raw available:",
    adata29.raw is not None
)

print(
    "Layers:",
    list(
        adata29.layers.keys()
    )
)

if sp.issparse(X):

    values = X.data

else:

    values = np.asarray(
        X
    ).ravel()

print(
    "\nStored values:"
)

print(
    "Min:",
    float(
        values.min()
    )
)

print(
    "Median non-zero:",
    float(
        np.median(values)
    )
)

print(
    "P99:",
    float(
        np.quantile(
            values,
            0.99
        )
    )
)

print(
    "Max:",
    float(
        values.max()
    )
)

In [ ]:
# =============================================================================
# Step29A Cell 9
# Raw-count-like value audit
# =============================================================================

sample_values = values

if len(sample_values) > 200000:

    rng = np.random.default_rng(
        42
    )

    sample_values = rng.choice(
        sample_values,
        size=200000,
        replace=False
    )

integer_like_fraction = np.mean(
    np.isclose(
        sample_values,
        np.round(
            sample_values
        ),
        atol=1e-8
    )
)

print(
    "Integer-like fraction:",
    integer_like_fraction
)

if integer_like_fraction > 0.99:

    print(
        "Matrix appears count-like."
    )

else:

    print(
        "Matrix appears transformed / normalized."
    )

In [ ]:
# =============================================================================
# Step29A Cell 10
# Gene identifier audit
# =============================================================================

genes = pd.Index(
    adata29.var_names.astype(str)
)

gene_audit = pd.DataFrame([
    {
        "N_genes":
            len(genes),

        "N_unique":
            genes.nunique(),

        "N_duplicates":
            int(
                genes.duplicated().sum()
            ),

        "N_Ensembl_like":
            int(
                genes.str.startswith(
                    "ENSG"
                ).sum()
            )
    }
])

display(
    gene_audit
)

print(
    "\nFirst 30 gene IDs:"
)

print(
    genes[:30].tolist()
)

assert (
    genes.duplicated().sum()
    == 0
)

In [ ]:
# =============================================================================
# Step29A Cell 11
# Frozen Core50 field audit
# =============================================================================

core50_score = (
    adata29.obs[
        "MS3_conserved_Core50_score"
    ]
    .to_numpy(
        dtype=float
    )
)

print(
    "Core50 score N:",
    len(core50_score)
)

print(
    "Missing:",
    int(
        np.isnan(
            core50_score
        ).sum()
    )
)

print(
    "Mean:",
    float(
        np.nanmean(
            core50_score
        )
    )
)

print(
    "SD:",
    float(
        np.nanstd(
            core50_score
        )
    )
)

print(
    "Range:",
    (
        float(
            np.nanmin(
                core50_score
            )
        ),
        float(
            np.nanmax(
                core50_score
            )
        )
    )
)

In [ ]:
# =============================================================================
# Step29A Cell 12
# Targeted input/resource audit
# =============================================================================

from pathlib import Path
import pandas as pd

# -------------------------------------------------------------------------
# Known expression objects
# -------------------------------------------------------------------------

known_h5ad = [
    ROOT /
    "LIHC_scRNA_Step27_Trajectory" /
    "Data" /
    "GSE149614_STEP27_FINAL_TRAJECTORY.h5ad",

    ROOT /
    "LIHC_scRNA_Step23_Malignant_State_Analysis" /
    "Data" /
    "GSE149614_STEP23D3_LINEAGE_CLEAN_RAW_COUNTS.h5ad"
]

candidate_rows = []

for path in known_h5ad:

    candidate_rows.append({
        "File":
            path.name,

        "Exists":
            path.exists(),

        "Path":
            str(path),

        "Size_MB":
            round(
                path.stat().st_size / 1024**2,
                2
            )
            if path.exists()
            else None
    })

candidate_h5ad_df = pd.DataFrame(
    candidate_rows
)

display(
    candidate_h5ad_df
)

In [ ]:
# =============================================================================
# Step29A Cell 12B
# Targeted GRN resource audit
# =============================================================================

RESOURCE_DIR = (
    ROOT /
    "LIHC_scRNA_Step29_GRN" /
    "Resources"
)

known_resources = [
    RESOURCE_DIR /
    "tf" /
    "allTFs_hg38.txt",

    RESOURCE_DIR /
    "motif" /
    "motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl",

    RESOURCE_DIR /
    "cistarget" /
    "hg38_10kbp_up_10kbp_down_full_tx_v10_clust."
    "genes_vs_motifs.rankings.feather",

    RESOURCE_DIR /
    "cistarget" /
    "hg38_500bp_up_100bp_down_full_tx_v10_clust."
    "genes_vs_motifs.rankings.feather"
]

resource_rows = []

for path in known_resources:

    resource_rows.append({
        "File":
            path.name,

        "Exists":
            path.exists(),

        "Path":
            str(path),

        "Suffix":
            path.suffix,

        "Size_MB":
            round(
                path.stat().st_size / 1024**2,
                2
            )
            if path.exists()
            else None
    })

grn_resource_df = pd.DataFrame(
    resource_rows
)

display(
    grn_resource_df
)

In [ ]:
# =============================================================================
# Step29A Cell 12C
# Targeted resource lock check
# =============================================================================

INPUT_RESOURCE_PASS = (
    candidate_h5ad_df["Exists"].all()
    and
    grn_resource_df["Exists"].all()
)

print(
    "Expression objects:",
    candidate_h5ad_df["Exists"].sum(),
    "/",
    len(candidate_h5ad_df)
)

print(
    "GRN resources:",
    grn_resource_df["Exists"].sum(),
    "/",
    len(grn_resource_df)
)

print(
    "\nINPUT_RESOURCE_PASS:",
    INPUT_RESOURCE_PASS
)

assert INPUT_RESOURCE_PASS

print(
    "\nTARGETED INPUT/RESOURCE AUDIT: PASS"
)

In [ ]:
# =============================================================================
# Step29A Cell 13
# GRN software preflight + REPORT
# =============================================================================

packages = [
    "pyscenic",
    "arboreto",
    "ctxcore",
    "dask",
    "distributed",
    "decoupler"
]

package_rows = []

for pkg in packages:

    spec = importlib.util.find_spec(
        pkg
    )

    package_rows.append({
        "Package":
            pkg,

        "Installed":
            spec is not None
    })

package_audit = pd.DataFrame(
    package_rows
)


print("=" * 100)

print(
    "STEP29A GRN PREFLIGHT REPORT"
)

print("=" * 100)


print(
    "\n[1] Frozen input"
)

print(
    STEP27_H5AD
)

print(
    "\nCells / genes:",
    adata29.n_obs,
    "/",
    adata29.n_vars
)


print(
    "\n[2] Frozen state counts"
)

display(
    state_counts.to_frame("N")
)


print(
    "\n[3] Patient × state"
)

display(
    patient_state
)


print(
    "\n[4] Expression matrix"
)

print(
    "X type:",
    type(
        adata29.X
    )
)

print(
    "Sparse:",
    sp.issparse(
        adata29.X
    )
)

print(
    "Raw available:",
    adata29.raw is not None
)

print(
    "Layers:",
    list(
        adata29.layers.keys()
    )
)

print(
    "Integer-like fraction:",
    integer_like_fraction
)


print(
    "\n[5] Gene audit"
)

display(
    gene_audit
)

print(
    "First 30 genes:"
)

print(
    genes[:30].tolist()
)


print(
    "\n[6] Candidate raw/scRNA h5ad objects"
)

display(
    candidate_h5ad_df
)


print(
    "\n[7] Existing GRN / motif resources"
)

display(
    grn_resource_df.head(100)
)


print(
    "\n[8] GRN software availability"
)

display(
    package_audit
)


print(
    "\n[9] Design lock"
)

print(
    "Malignant-cell universe remains frozen at 12,057 cells."
)

print(
    "MS1-MS8 will not be reclustered or redefined."
)

print(
    "Core50 and pseudotime will not be used "
    "to select genes for initial GRN inference."
)

print(
    "They will be used only for downstream "
    "regulon interpretation and validation."
)

print(
    "\nSTEP29A PREFLIGHT COMPLETE"
)

In [ ]:
# =============================================================================
# Step29B Cell 14
# Locate the lineage-clean raw-count companion object
# =============================================================================

raw_hits = list(
    ROOT.rglob(
        "GSE149614_STEP23D3_LINEAGE_CLEAN_RAW_COUNTS.h5ad"
    )
)

print(
    "Number of matches:",
    len(raw_hits)
)

for p in raw_hits:
    print(p)

assert len(raw_hits) == 1

RAW_H5AD = raw_hits[0]

print(
    "\nRAW candidate locked:"
)

print(
    RAW_H5AD
)

In [ ]:
# =============================================================================
# Step29B Cell 15
# Load raw-count companion object
# =============================================================================

adata_raw29 = sc.read_h5ad(
    RAW_H5AD
)

print(
    "Raw object shape:",
    adata_raw29.shape
)

print(
    "X type:",
    type(
        adata_raw29.X
    )
)

print(
    "Sparse:",
    sp.issparse(
        adata_raw29.X
    )
)

print(
    "Raw slot:",
    adata_raw29.raw is not None
)

print(
    "Layers:",
    list(
        adata_raw29.layers.keys()
    )
)

In [ ]:
# =============================================================================
# Step29B Cell 16
# Cell identity audit against frozen Step27 universe
# =============================================================================

frozen_cells = pd.Index(
    adata29.obs_names.astype(str)
)

raw_cells = pd.Index(
    adata_raw29.obs_names.astype(str)
)

same_n_cells = (
    len(frozen_cells)
    == len(raw_cells)
)

same_cell_set = (
    set(frozen_cells)
    == set(raw_cells)
)

same_cell_order = np.array_equal(
    frozen_cells.to_numpy(),
    raw_cells.to_numpy()
)

missing_in_raw = (
    frozen_cells
    .difference(raw_cells)
)

extra_in_raw = (
    raw_cells
    .difference(frozen_cells)
)

print(
    "Frozen N:",
    len(frozen_cells)
)

print(
    "Raw N:",
    len(raw_cells)
)

print(
    "Same N:",
    same_n_cells
)

print(
    "Same cell set:",
    same_cell_set
)

print(
    "Same cell order:",
    same_cell_order
)

print(
    "Missing in raw:",
    len(missing_in_raw)
)

print(
    "Extra in raw:",
    len(extra_in_raw)
)

In [ ]:
# =============================================================================
# Step29B Cell 17
# Gene identity audit
# =============================================================================

frozen_genes = pd.Index(
    adata29.var_names.astype(str)
)

raw_genes = pd.Index(
    adata_raw29.var_names.astype(str)
)

same_n_genes = (
    len(frozen_genes)
    == len(raw_genes)
)

same_gene_set = (
    set(frozen_genes)
    == set(raw_genes)
)

same_gene_order = np.array_equal(
    frozen_genes.to_numpy(),
    raw_genes.to_numpy()
)

gene_overlap = (
    frozen_genes
    .intersection(raw_genes)
)

print(
    "Frozen genes:",
    len(frozen_genes)
)

print(
    "Raw genes:",
    len(raw_genes)
)

print(
    "Same N:",
    same_n_genes
)

print(
    "Same gene set:",
    same_gene_set
)

print(
    "Same gene order:",
    same_gene_order
)

print(
    "Gene overlap:",
    len(gene_overlap)
)

print(
    "Raw duplicated genes:",
    int(
        raw_genes
        .duplicated()
        .sum()
    )
)

In [ ]:
# =============================================================================
# Step29B Cell 18
# Raw-count value audit
# =============================================================================

Xraw = adata_raw29.X

if sp.issparse(Xraw):

    raw_values = Xraw.data

else:

    raw_values = np.asarray(
        Xraw
    ).ravel()


rng = np.random.default_rng(
    42
)

if len(raw_values) > 200000:

    sampled_raw_values = rng.choice(
        raw_values,
        size=200000,
        replace=False
    )

else:

    sampled_raw_values = raw_values


raw_integer_fraction = np.mean(
    np.isclose(
        sampled_raw_values,
        np.round(
            sampled_raw_values
        ),
        atol=1e-8
    )
)

print(
    "Min:",
    float(
        raw_values.min()
    )
)

print(
    "Median nonzero:",
    float(
        np.median(
            raw_values
        )
    )
)

print(
    "P99:",
    float(
        np.quantile(
            raw_values,
            0.99
        )
    )
)

print(
    "Max:",
    float(
        raw_values.max()
    )
)

print(
    "Integer-like fraction:",
    raw_integer_fraction
)

print(
    "Any negative:",
    bool(
        np.any(
            raw_values < 0
        )
    )
)

In [ ]:
# =============================================================================
# Step29B Cell 19
# Count matrix basic QC
# =============================================================================

if sp.issparse(Xraw):

    counts_per_cell = np.asarray(
        Xraw.sum(axis=1)
    ).ravel()

    detected_per_cell = np.asarray(
        (Xraw > 0).sum(axis=1)
    ).ravel()

    detected_per_gene = np.asarray(
        (Xraw > 0).sum(axis=0)
    ).ravel()

else:

    counts_per_cell = (
        Xraw.sum(axis=1)
    )

    detected_per_cell = (
        (Xraw > 0)
        .sum(axis=1)
    )

    detected_per_gene = (
        (Xraw > 0)
        .sum(axis=0)
    )


print(
    "Counts/cell median:",
    float(
        np.median(
            counts_per_cell
        )
    )
)

print(
    "Detected genes/cell median:",
    float(
        np.median(
            detected_per_cell
        )
    )
)

print(
    "Genes detected >=1 cell:",
    int(
        np.sum(
            detected_per_gene >= 1
        )
    )
)

print(
    "Genes detected >=1% cells:",
    int(
        np.sum(
            detected_per_gene
            >= 0.01 * adata_raw29.n_obs
        )
    )
)

print(
    "Genes detected >=3% cells:",
    int(
        np.sum(
            detected_per_gene
            >= 0.03 * adata_raw29.n_obs
        )
    )
)

In [ ]:
# =============================================================================
# Step29B Cell 20
# REPORT — Raw-count input lock
# =============================================================================

RAW_INPUT_PASS = (
    adata_raw29.n_obs == 12057
    and
    same_cell_set
    and
    raw_integer_fraction > 0.999
    and
    not np.any(raw_values < 0)
    and
    raw_genes.duplicated().sum() == 0
)

print(
    "=" * 100
)

print(
    "STEP29B RAW-COUNT INPUT LOCK REPORT"
)

print(
    "=" * 100
)

print(
    "\nRaw object:"
)

print(
    RAW_H5AD
)

print(
    "\nShape:",
    adata_raw29.shape
)

print(
    "\nCell identity:"
)

print(
    "Same cell set:",
    same_cell_set
)

print(
    "Same cell order:",
    same_cell_order
)

print(
    "Missing cells:",
    len(
        missing_in_raw
    )
)

print(
    "Extra cells:",
    len(
        extra_in_raw
    )
)

print(
    "\nGene identity:"
)

print(
    "Same gene set:",
    same_gene_set
)

print(
    "Same gene order:",
    same_gene_order
)

print(
    "Gene overlap:",
    len(
        gene_overlap
    )
)

print(
    "\nCount validity:"
)

print(
    "Integer-like fraction:",
    raw_integer_fraction
)

print(
    "Negative values:",
    bool(
        np.any(
            raw_values < 0
        )
    )
)

print(
    "\nDetection:"
)

print(
    "Genes >=1% cells:",
    int(
        np.sum(
            detected_per_gene
            >= 0.01 * adata_raw29.n_obs
        )
    )
)

print(
    "Genes >=3% cells:",
    int(
        np.sum(
            detected_per_gene
            >= 0.03 * adata_raw29.n_obs
        )
    )
)

print(
    "\nRAW_INPUT_PASS:",
    RAW_INPUT_PASS
)

print(
    "\nSTEP29B RAW INPUT AUDIT COMPLETE"
)

In [ ]:
import sys
import pyscenic
import arboreto
import ctxcore

print("Python:", sys.executable)
print("pySCENIC:", pyscenic.__version__)
print("STEP29 KERNEL: PASS")

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step29C Cell 21
# Resource directories
# =============================================================================

from pathlib import Path
import sys
import importlib.metadata as md

ROOT = Path(_hcc_path(''))

STEP29_DIR = ROOT / "LIHC_scRNA_Step29_GRN"
RESOURCE_DIR = STEP29_DIR / "Resources"

TF_DIR = RESOURCE_DIR / "tf"
MOTIF_DIR = RESOURCE_DIR / "motif"
CISTARGET_DIR = RESOURCE_DIR / "cistarget"

for p in [
    TF_DIR,
    MOTIF_DIR,
    CISTARGET_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

print("Python:", sys.executable)
print("pySCENIC:", md.version("pyscenic"))

print("\nTF_DIR:", TF_DIR)
print("MOTIF_DIR:", MOTIF_DIR)
print("CISTARGET_DIR:", CISTARGET_DIR)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step29C Cell 22
# Resource file audit
# =============================================================================

from pathlib import Path
import pandas as pd

ROOT = Path(_hcc_path(''))
STEP29_DIR = ROOT / "LIHC_scRNA_Step29_GRN"

RESOURCE_DIR = STEP29_DIR / "Resources"
TF_DIR = RESOURCE_DIR / "tf"
MOTIF_DIR = RESOURCE_DIR / "motif"
CISTARGET_DIR = RESOURCE_DIR / "cistarget"

TF_FILE = TF_DIR / "allTFs_hg38.txt"

MOTIF_FILE = (
    MOTIF_DIR /
    "motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl"
)

DB_10KB = (
    CISTARGET_DIR /
    "hg38_10kbp_up_10kbp_down_full_tx_v10_clust."
    "genes_vs_motifs.rankings.feather"
)

DB_500BP = (
    CISTARGET_DIR /
    "hg38_500bp_up_100bp_down_full_tx_v10_clust."
    "genes_vs_motifs.rankings.feather"
)

resource_files = {
    "TF_list": TF_FILE,
    "Motif_annotation": MOTIF_FILE,
    "Ranking_10kb": DB_10KB,
    "Ranking_500bp": DB_500BP
}

rows = []

for name, path in resource_files.items():

    rows.append({
        "Resource": name,
        "Exists": path.exists(),
        "Size_MB":
            round(
                path.stat().st_size / 1024**2,
                2
            )
            if path.exists()
            else None,
        "Path": str(path)
    })

resource_audit = pd.DataFrame(rows)

display(resource_audit)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step29C Recovery Cell
# Reload frozen objects after kernel switch
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc

ROOT = Path(_hcc_path(''))

STEP29_DIR = ROOT / "LIHC_scRNA_Step29_GRN"
RESOURCE_DIR = STEP29_DIR / "Resources"

TF_DIR = RESOURCE_DIR / "tf"
MOTIF_DIR = RESOURCE_DIR / "motif"
CISTARGET_DIR = RESOURCE_DIR / "cistarget"

STEP27_H5AD = (
    ROOT /
    "LIHC_scRNA_Step27_Trajectory" /
    "Data" /
    "GSE149614_STEP27_FINAL_TRAJECTORY.h5ad"
)

RAW_H5AD = (
    ROOT /
    "LIHC_scRNA_Step23_Malignant_State_Analysis" /
    "Data" /
    "GSE149614_STEP23D3_LINEAGE_CLEAN_RAW_COUNTS.h5ad"
)

adata29 = sc.read_h5ad(
    STEP27_H5AD
)

adata_raw29 = sc.read_h5ad(
    RAW_H5AD
)

print(
    "Frozen object:",
    adata29.shape
)

print(
    "Raw-count object:",
    adata_raw29.shape
)

print(
    "Same cells:",
    np.array_equal(
        adata29.obs_names,
        adata_raw29.obs_names
    )
)

print(
    "Same genes:",
    np.array_equal(
        adata29.var_names,
        adata_raw29.var_names
    )
)

assert adata29.shape == (12057, 25712)
assert adata_raw29.shape == (12057, 25712)

print(
    "\nSTEP29 CORE OBJECTS RELOADED: PASS"
)

In [ ]:
# =============================================================================
# Step29C Resource paths
# =============================================================================

TF_FILE = (
    TF_DIR /
    "allTFs_hg38.txt"
)

MOTIF_FILE = (
    MOTIF_DIR /
    "motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl"
)

DB_10KB = (
    CISTARGET_DIR /
    "hg38_10kbp_up_10kbp_down_full_tx_v10_clust."
    "genes_vs_motifs.rankings.feather"
)

DB_500BP = (
    CISTARGET_DIR /
    "hg38_500bp_up_100bp_down_full_tx_v10_clust."
    "genes_vs_motifs.rankings.feather"
)

for x in [
    TF_FILE,
    MOTIF_FILE,
    DB_10KB,
    DB_500BP
]:
    print(
        x.name,
        "->",
        x.exists()
    )

assert all(
    x.exists()
    for x in [
        TF_FILE,
        MOTIF_FILE,
        DB_10KB,
        DB_500BP
    ]
)

print(
    "\nALL GRN RESOURCES: PASS"
)

In [ ]:
# =============================================================================
# Step29C Cell 23
# Human TF overlap with frozen expression genes
# =============================================================================

with open(
    TF_FILE,
    "r"
) as f:

    tf_list = [
        line.strip()
        for line in f
        if line.strip()
    ]

tf_list = sorted(
    set(tf_list)
)

expr_genes = set(
    adata29.var_names.astype(str)
)

tf_overlap = sorted(
    set(tf_list)
    & expr_genes
)

print(
    "TFs in reference:",
    len(tf_list)
)

print(
    "TFs present in expression matrix:",
    len(tf_overlap)
)

print(
    "TF overlap fraction:",
    len(tf_overlap)
    / len(tf_list)
)

print(
    "\nFirst 30 overlapping TFs:"
)

print(
    tf_overlap[:30]
)

In [ ]:
# =============================================================================
# Step29C Cell 24
# cisTarget database compatibility test
# =============================================================================

from ctxcore.rnkdb import FeatherRankingDatabase

assert DB_10KB.exists()
assert DB_500BP.exists()
assert MOTIF_FILE.exists()

db10 = FeatherRankingDatabase(
    fname=str(DB_10KB),
    name="hg38_10kb"
)

db500 = FeatherRankingDatabase(
    fname=str(DB_500BP),
    name="hg38_500bp"
)

print(
    "10kb database:",
    db10
)

print(
    "\n500bp database:",
    db500
)

print(
    "\nCISTARGET DATABASE LOAD: PASS"
)

In [ ]:
# =============================================================================
# Step29C Cell 25
# REPORT — Resource lock
# =============================================================================

import sys
import importlib.metadata as md

RESOURCE_PASS = (
    resource_audit["Exists"].all()
    and len(tf_overlap) > 0
)

print("=" * 100)
print("STEP29C GRN RESOURCE LOCK REPORT")
print("=" * 100)

print(
    "\nPython:",
    sys.executable
)

print(
    "pySCENIC:",
    md.version("pyscenic")
)

print(
    "\n[1] Resources"
)

display(
    resource_audit
)

print(
    "\n[2] TF overlap"
)

print(
    "Reference TFs:",
    len(tf_list)
)

print(
    "Expression-matched TFs:",
    len(tf_overlap)
)

print(
    "Overlap fraction:",
    round(
        len(tf_overlap)
        / len(tf_list),
        4
    )
)

print(
    "\n[3] cisTarget"
)

print(
    "10kb database: loaded"
)

print(
    "500bp database: loaded"
)

print(
    "\nRESOURCE_PASS:",
    RESOURCE_PASS
)

print(
    "\nSTEP29C RESOURCE LOCK COMPLETE"
)

In [ ]:
# =============================================================================
# Step29D Cell 26
# Pre-registered GRN gene-filtering thresholds
# =============================================================================

import math

N_CELLS = adata_raw29.n_obs

MIN_CELL_FRACTION = 0.01

MIN_CELLS = math.ceil(
    N_CELLS * MIN_CELL_FRACTION
)

MIN_COUNTS = math.ceil(
    3 * N_CELLS * MIN_CELL_FRACTION
)

print(
    "N cells:",
    N_CELLS
)

print(
    "Minimum detected cells:",
    MIN_CELLS
)

print(
    "Minimum total counts:",
    MIN_COUNTS
)

assert N_CELLS == 12057
assert MIN_CELLS == 121
assert MIN_COUNTS == 362

In [ ]:
# =============================================================================
# Step29D Cell 27
# Per-gene expression QC from frozen raw counts
# =============================================================================

Xraw = adata_raw29.X

if sp.issparse(Xraw):

    gene_total_counts = np.asarray(
        Xraw.sum(axis=0)
    ).ravel()

    gene_detected_cells = np.asarray(
        (Xraw > 0).sum(axis=0)
    ).ravel()

else:

    gene_total_counts = np.asarray(
        Xraw.sum(axis=0)
    ).ravel()

    gene_detected_cells = np.asarray(
        (Xraw > 0).sum(axis=0)
    ).ravel()


gene_qc = pd.DataFrame({
    "Gene":
        adata_raw29.var_names.astype(str),

    "Total_counts":
        gene_total_counts,

    "Detected_cells":
        gene_detected_cells
})

gene_qc[
    "Detected_fraction"
] = (
    gene_qc["Detected_cells"]
    / N_CELLS
)

gene_qc[
    "Pass_expression_filter"
] = (
    (gene_qc["Detected_cells"] >= MIN_CELLS)
    &
    (gene_qc["Total_counts"] >= MIN_COUNTS)
)

print(
    "Genes total:",
    len(gene_qc)
)

print(
    "Genes passing expression filter:",
    int(
        gene_qc[
            "Pass_expression_filter"
        ].sum()
    )
)

In [ ]:
# =============================================================================
# Step29D Cell 28
# cisTarget-compatible gene universe
# =============================================================================

db10_genes = set(
    map(
        str,
        db10.genes
    )
)

db500_genes = set(
    map(
        str,
        db500.genes
    )
)

db_common_genes = (
    db10_genes
    & db500_genes
)

print(
    "10kb database genes:",
    len(db10_genes)
)

print(
    "500bp database genes:",
    len(db500_genes)
)

print(
    "Common database genes:",
    len(db_common_genes)
)

gene_qc[
    "In_both_cisTarget_DBs"
] = (
    gene_qc["Gene"]
    .isin(
        db_common_genes
    )
)

gene_qc[
    "GRN_keep"
] = (
    gene_qc[
        "Pass_expression_filter"
    ]
    &
    gene_qc[
        "In_both_cisTarget_DBs"
    ]
)

grn_genes = (
    gene_qc.loc[
        gene_qc["GRN_keep"],
        "Gene"
    ]
    .tolist()
)

print(
    "\nFinal GRN genes:",
    len(grn_genes)
)

In [ ]:
# =============================================================================
# Step29D Cell 29
# TF coverage after GRN gene filtering
# =============================================================================

grn_gene_set = set(
    grn_genes
)

grn_tfs = sorted(
    set(tf_list)
    & grn_gene_set
)

print(
    "Reference TFs:",
    len(tf_list)
)

print(
    "TFs in original expression:",
    len(tf_overlap)
)

print(
    "TFs retained for GRN:",
    len(grn_tfs)
)

print(
    "Retained TF fraction:",
    len(grn_tfs)
    / len(tf_list)
)

In [ ]:
# =============================================================================
# Step29D Cell 30
# Sentinel TF audit — descriptive only
# =============================================================================

sentinel_tfs = [
    "HNF4A",
    "HNF1A",
    "FOXM1",
    "E2F1",
    "E2F7",
    "E2F8",
    "MYBL2",
    "MYC",
    "STAT3",
    "SMAD3"
]

sentinel_audit = []

for tf in sentinel_tfs:

    row = gene_qc.loc[
        gene_qc["Gene"] == tf
    ]

    if len(row) == 0:

        sentinel_audit.append({
            "TF": tf,
            "Present": False,
            "Pass_expression": False,
            "In_cisTarget": False,
            "GRN_keep": False
        })

    else:

        r = row.iloc[0]

        sentinel_audit.append({
            "TF":
                tf,

            "Present":
                True,

            "Detected_cells":
                int(
                    r["Detected_cells"]
                ),

            "Detected_fraction":
                float(
                    r[
                        "Detected_fraction"
                    ]
                ),

            "Pass_expression":
                bool(
                    r[
                        "Pass_expression_filter"
                    ]
                ),

            "In_cisTarget":
                bool(
                    r[
                        "In_both_cisTarget_DBs"
                    ]
                ),

            "GRN_keep":
                bool(
                    r[
                        "GRN_keep"
                    ]
                )
        })

sentinel_audit = pd.DataFrame(
    sentinel_audit
)

display(
    sentinel_audit
)

In [ ]:
# =============================================================================
# Step29D Cell 31
# Build clean GRN inference object
# =============================================================================

import anndata as ad

gene_mask = (
    adata_raw29.var_names
    .astype(str)
    .isin(
        grn_genes
    )
)

X_grn = (
    adata_raw29[
        :,
        gene_mask
    ].X.copy()
)

grn_var_names = (
    adata_raw29.var_names[
        gene_mask
    ].astype(str)
)

adata_grn = ad.AnnData(
    X=X_grn,

    obs=pd.DataFrame(
        index=adata_raw29.obs_names.copy()
    ),

    var=pd.DataFrame(
        index=grn_var_names
    )
)

print(
    "GRN object:",
    adata_grn.shape
)

print(
    "Sparse:",
    sp.issparse(
        adata_grn.X
    )
)

In [ ]:
# =============================================================================
# Step29D Cell 32
# Save frozen biological metadata separately
# =============================================================================

metadata_cols = [
    "patient",
    "final_malignant_state_id",
    "MS6_anchored_pseudotime",
    "MS3_conserved_Core50_score",
    "Program__Proliferation_G2M",
    "DeepProgram__Mitotic_Entry",
    "DeepProgram__Spindle_Chromosome_Segregation",
    "DeepProgram__Hepatic_Differentiation",
    "DeepProgram__Bile_Acid_Metabolism"
]

metadata29 = (
    adata29.obs[
        metadata_cols
    ]
    .copy()
)

metadata29.insert(
    0,
    "Cell_ID",
    metadata29.index
)

print(
    "Metadata shape:",
    metadata29.shape
)

display(
    metadata29.head()
)

In [ ]:
# =============================================================================
# Step29D Cell 33
# Save GRN input lock
# =============================================================================

DATA_DIR = (
    STEP29_DIR /
    "Data"
)

TABLE_DIR = (
    STEP29_DIR /
    "Tables"
)

AUDIT_DIR = (
    STEP29_DIR /
    "Audit"
)

for p in [
    DATA_DIR,
    TABLE_DIR,
    AUDIT_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

GRN_INPUT_H5AD = (
    DATA_DIR /
    "Step29D_GRN_INPUT_FILTERED_RAW_COUNTS.h5ad"
)

adata_grn.write_h5ad(
    GRN_INPUT_H5AD,
    compression="gzip"
)

gene_qc.to_csv(
    TABLE_DIR /
    "Step29D_Gene_Filtering_Audit.csv",
    index=False
)

pd.Series(
    grn_genes,
    name="Gene"
).to_csv(
    TABLE_DIR /
    "Step29D_GRN_Genes.csv",
    index=False
)

pd.Series(
    grn_tfs,
    name="TF"
).to_csv(
    TABLE_DIR /
    "Step29D_GRN_TFs.csv",
    index=False
)

sentinel_audit.to_csv(
    TABLE_DIR /
    "Step29D_Sentinel_TF_Audit.csv",
    index=False
)

metadata29.to_csv(
    TABLE_DIR /
    "Step29D_Frozen_Cell_Metadata.csv",
    index=False
)

print(
    "Saved:",
    GRN_INPUT_H5AD
)

In [ ]:
# =============================================================================
# Step29D Cell 34
# REPORT — GRN expression input lock
# =============================================================================

GRN_INPUT_PASS = (
    adata_grn.n_obs == 12057
    and
    adata_grn.n_vars > 0
    and
    len(grn_tfs) > 0
    and
    np.array_equal(
        adata_grn.obs_names,
        adata_raw29.obs_names
    )
)

print("=" * 100)

print(
    "STEP29D GRN EXPRESSION INPUT LOCK REPORT"
)

print("=" * 100)

print(
    "\nCells:",
    adata_grn.n_obs
)

print(
    "Original genes:",
    adata_raw29.n_vars
)

print(
    "Expression-filtered genes:",
    int(
        gene_qc[
            "Pass_expression_filter"
        ].sum()
    )
)

print(
    "cisTarget-compatible genes:",
    int(
        gene_qc[
            "In_both_cisTarget_DBs"
        ].sum()
    )
)

print(
    "Final GRN genes:",
    adata_grn.n_vars
)

print(
    "\nReference TFs:",
    len(tf_list)
)

print(
    "Final GRN TFs:",
    len(grn_tfs)
)

print(
    "\nThresholds:"
)

print(
    "Min detected cells:",
    MIN_CELLS
)

print(
    "Min total counts:",
    MIN_COUNTS
)

print(
    "\nSentinel TF audit:"
)

display(
    sentinel_audit
)

print(
    "\nInput file:"
)

print(
    GRN_INPUT_H5AD
)

print(
    "\nGRN_INPUT_PASS:",
    GRN_INPUT_PASS
)

print(
    "\nDesign boundary:"
)

print(
    "No HVG, DEG, Core50, state-label, "
    "or pseudotime-based gene selection was used."
)

print(
    "Frozen biological annotations were saved "
    "separately and were not used for GRN inference."
)

print(
    "\nSTEP29D INPUT LOCK COMPLETE"
)

In [ ]:
# =============================================================================
# Step29E Cell 35
# Final pySCENIC CLI preflight
# =============================================================================

import sys
import subprocess
from pathlib import Path

PYSCENIC_CLI = (
    Path(sys.executable).parent /
    "pyscenic"
)

tests = [
    [str(PYSCENIC_CLI), "-h"],
    [str(PYSCENIC_CLI), "grn", "-h"]
]

results = []

for cmd in tests:

    res = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    results.append(
        res.returncode
    )

    print(
        "COMMAND:",
        " ".join(cmd)
    )

    print(
        "Return code:",
        res.returncode
    )

    if res.stderr:
        print(
            "STDERR:",
            res.stderr[:2000]
        )

    print("-" * 80)

CLI_PASS = all(
    x == 0
    for x in results
)

print(
    "PYSCENIC CLI PREFLIGHT:",
    CLI_PASS
)

assert CLI_PASS

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step29E Cell 36
# Restore frozen Step29D input and derive log2(count+1) GRN matrix
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad

ROOT = Path(_hcc_path(''))

STEP29_DIR = ROOT / "LIHC_scRNA_Step29_GRN"
DATA_DIR = STEP29_DIR / "Data"
TABLE_DIR = STEP29_DIR / "Tables"
AUDIT_DIR = STEP29_DIR / "Audit"

GRN_RAW_H5AD = (
    DATA_DIR /
    "Step29D_GRN_INPUT_FILTERED_RAW_COUNTS.h5ad"
)

GRN_TF_CSV = (
    TABLE_DIR /
    "Step29D_GRN_TFs.csv"
)

assert GRN_RAW_H5AD.exists()
assert GRN_TF_CSV.exists()

adata_grn_raw = sc.read_h5ad(
    GRN_RAW_H5AD
)

grn_tfs = (
    pd.read_csv(GRN_TF_CSV)["TF"]
    .astype(str)
    .tolist()
)

print(
    "Raw GRN object:",
    adata_grn_raw.shape
)

print(
    "TFs:",
    len(grn_tfs)
)

assert adata_grn_raw.shape == (
    12057,
    12916
)

assert len(grn_tfs) == 1329


X_log = (
    adata_grn_raw.X
    .astype(np.float32)
    .copy()
)

if sp.issparse(X_log):

    X_log = X_log.tocsr()

    X_log.data = np.log2(
        X_log.data + 1.0
    )

else:

    X_log = np.log2(
        X_log + 1.0
    ).astype(
        np.float32
    )


adata_grn_log = ad.AnnData(
    X=X_log,

    obs=pd.DataFrame(
        index=
        adata_grn_raw.obs_names.copy()
    ),

    var=pd.DataFrame(
        index=
        adata_grn_raw.var_names.copy()
    )
)

print(
    "\nLog2 GRN object:",
    adata_grn_log.shape
)

print(
    "Sparse:",
    sp.issparse(
        adata_grn_log.X
    )
)

print(
    "Finite:",
    bool(
        np.isfinite(
            adata_grn_log.X.data
        ).all()
    )
)

In [ ]:
# =============================================================================
# Step29E Cell 37
# Save derived GRNBoost2 expression matrix
# =============================================================================

GRN_LOG_H5AD = (
    DATA_DIR /
    "Step29E_GRN_INPUT_LOG2_COUNTS.h5ad"
)

if not GRN_LOG_H5AD.exists():

    adata_grn_log.write_h5ad(
        GRN_LOG_H5AD,
        compression="gzip"
    )

print(
    "Saved:",
    GRN_LOG_H5AD
)

print(
    "Size MB:",
    round(
        GRN_LOG_H5AD.stat().st_size
        / 1024**2,
        2
    )
)

In [ ]:
# =============================================================================
# Step29E Cell 38
# Save formal GRN TF list
# =============================================================================

GRN_TF_TXT = (
    DATA_DIR /
    "Step29E_GRN_TFs_1329.txt"
)

with open(
    GRN_TF_TXT,
    "w"
) as f:

    for tf in grn_tfs:

        f.write(
            tf + "\n"
        )

print(
    "TF file:",
    GRN_TF_TXT
)

print(
    "N TFs:",
    len(grn_tfs)
)

In [ ]:
# =============================================================================
# Step29E Cell 39
# Create sparse loom for GRNBoost2
# =============================================================================

import loompy

GRN_LOOM = (
    DATA_DIR /
    "Step29E_GRN_INPUT_LOG2_COUNTS.loom"
)

X_loom = (
    adata_grn_log.X.T
)

if not sp.issparse(
    X_loom
):

    X_loom = sp.csr_matrix(
        X_loom
    )

X_loom = X_loom.tocsr()


if GRN_LOOM.exists():

    print(
        "Existing loom detected:"
    )

    print(
        GRN_LOOM
    )

else:

    loompy.create(
        str(GRN_LOOM),

        X_loom,

        row_attrs={
            "Gene":
                np.asarray(
                    adata_grn_log.var_names,
                    dtype=str
                )
        },

        col_attrs={
            "CellID":
                np.asarray(
                    adata_grn_log.obs_names,
                    dtype=str
                )
        }
    )

print(
    "GRN loom:",
    GRN_LOOM
)

print(
    "Size MB:",
    round(
        GRN_LOOM.stat().st_size
        / 1024**2,
        2
    )
)

In [ ]:
# =============================================================================
# Step29E Cell 40
# Validate formal GRN loom
# =============================================================================

with loompy.connect(
    str(GRN_LOOM),
    mode="r"
) as ds:

    loom_shape = ds.shape

    loom_genes = (
        ds.ra["Gene"]
        .astype(str)
    )

    loom_cells = (
        ds.ca["CellID"]
        .astype(str)
    )

    print(
        "Shape:",
        loom_shape
    )

    print(
        "Row attrs:",
        list(
            ds.ra.keys()
        )
    )

    print(
        "Column attrs:",
        list(
            ds.ca.keys()
        )
    )

    print(
        "\nFirst genes:",
        loom_genes[:10]
    )

    print(
        "\nFirst cells:",
        loom_cells[:5]
    )

assert loom_shape == (
    12916,
    12057
)

assert np.array_equal(
    loom_genes,
    adata_grn_log.var_names.astype(str)
)

assert np.array_equal(
    loom_cells,
    adata_grn_log.obs_names.astype(str)
)

print(
    "\nFORMAL GRN LOOM: PASS"
)

In [ ]:
# =============================================================================
# Step29E Cell 41
# Multiprocessing Arboreto helper preflight
# =============================================================================

import sys
import subprocess

cmd = [
    sys.executable,
    "-m",
    "pyscenic.cli.arboreto_with_multiprocessing",
    "-h"
]

res = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(
    "Return code:",
    res.returncode
)

print(
    "\nSTDOUT:"
)

print(
    res.stdout[:5000]
)

print(
    "\nSTDERR:"
)

print(
    res.stderr[:2000]
)

assert res.returncode == 0

print(
    "\nARBORETO MULTIPROCESSING CLI: PASS"
)

In [ ]:
# =============================================================================
# Step29E Cell 42
# Build GRNBoost2 smoke-test dataset
# =============================================================================

rng = np.random.default_rng(
    42
)

smoke_cell_idx = np.sort(
    rng.choice(
        adata_grn_log.n_obs,
        size=500,
        replace=False
    )
)

smoke_tfs = (
    grn_tfs[:100]
)

other_genes = [
    g
    for g in
    adata_grn_log.var_names.astype(str)
    if g not in set(
        smoke_tfs
    )
]

smoke_genes = (
    smoke_tfs
    +
    other_genes[:900]
)

smoke_gene_idx = (
    adata_grn_log.var_names
    .get_indexer(
        smoke_genes
    )
)

assert np.all(
    smoke_gene_idx >= 0
)

SMOKE_LOOM = (
    DATA_DIR /
    "Step29E_SMOKE_GRN.loom"
)

SMOKE_TF_TXT = (
    DATA_DIR /
    "Step29E_SMOKE_TFs.txt"
)

X_smoke = (
    adata_grn_log.X[
        smoke_cell_idx
    ][:, smoke_gene_idx]
    .T
    .tocsr()
)

if SMOKE_LOOM.exists():
    SMOKE_LOOM.unlink()

loompy.create(
    str(SMOKE_LOOM),

    X_smoke,

    row_attrs={
        "Gene":
            np.asarray(
                smoke_genes,
                dtype=str
            )
    },

    col_attrs={
        "CellID":
            np.asarray(
                adata_grn_log.obs_names[
                    smoke_cell_idx
                ],
                dtype=str
            )
    }
)

with open(
    SMOKE_TF_TXT,
    "w"
) as f:

    for tf in smoke_tfs:

        f.write(
            tf + "\n"
        )

print(
    "Smoke matrix:",
    X_smoke.shape
)

print(
    "Smoke TFs:",
    len(smoke_tfs)
)

In [ ]:
# =============================================================================
# Step29E Cell 43
# Smoke-test GRNBoost2
# =============================================================================

SMOKE_ADJ = (
    TABLE_DIR /
    "Step29E_SMOKE_Adjacencies.tsv"
)

SMOKE_LOG = (
    AUDIT_DIR /
    "Step29E_SMOKE_GRNBoost2.log"
)

if SMOKE_ADJ.exists():
    SMOKE_ADJ.unlink()

cmd_smoke = [
    sys.executable,
    "-m",
    "pyscenic.cli.arboreto_with_multiprocessing",

    str(SMOKE_LOOM),
    str(SMOKE_TF_TXT),

    "--method",
    "grnboost2",

    "--sparse",

    "--num_workers",
    "2",

    "--seed",
    "42",

    "--gene_attribute",
    "Gene",

    "--cell_id_attribute",
    "CellID",

    "-o",
    str(SMOKE_ADJ)
]

print(
    "Running smoke GRNBoost2..."
)

with open(
    SMOKE_LOG,
    "w"
) as log:

    result = subprocess.run(
        cmd_smoke,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True
    )

print(
    "Return code:",
    result.returncode
)

print(
    "Output exists:",
    SMOKE_ADJ.exists()
)

print(
    "Log:",
    SMOKE_LOG
)

if result.returncode != 0:

    print(
        "\nLast 100 log lines:"
    )

    lines = (
        SMOKE_LOG
        .read_text(
            errors="ignore"
        )
        .splitlines()
    )

    print(
        "\n".join(
            lines[-100:]
        )
    )

assert result.returncode == 0
assert SMOKE_ADJ.exists()

print(
    "\nSMOKE GRNBOOST2: PASS"
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step29E Recovery after SciPy compatibility fix
# =============================================================================

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import scanpy as sc

ROOT = Path(_hcc_path(''))
STEP29_DIR = ROOT / "LIHC_scRNA_Step29_GRN"

DATA_DIR = STEP29_DIR / "Data"
TABLE_DIR = STEP29_DIR / "Tables"
AUDIT_DIR = STEP29_DIR / "Audit"

GRN_LOG_H5AD = (
    DATA_DIR /
    "Step29E_GRN_INPUT_LOG2_COUNTS.h5ad"
)

GRN_TF_TXT = (
    DATA_DIR /
    "Step29E_GRN_TFs_1329.txt"
)

GRN_LOOM = (
    DATA_DIR /
    "Step29E_GRN_INPUT_LOG2_COUNTS.loom"
)

SMOKE_LOOM = (
    DATA_DIR /
    "Step29E_SMOKE_GRN.loom"
)

SMOKE_TF_TXT = (
    DATA_DIR /
    "Step29E_SMOKE_TFs.txt"
)

SMOKE_ADJ = (
    TABLE_DIR /
    "Step29E_SMOKE_Adjacencies.tsv"
)

SMOKE_LOG = (
    AUDIT_DIR /
    "Step29E_SMOKE_GRNBoost2.log"
)

adata_grn_log = sc.read_h5ad(
    GRN_LOG_H5AD
)

with open(GRN_TF_TXT) as f:
    grn_tfs = [
        x.strip()
        for x in f
        if x.strip()
    ]

test_sparse = sp.csc_matrix(
    [[1, 0], [0, 1]]
)

print("Python:", sys.executable)
print("SciPy:", scipy.__version__)
print("Sparse .A:", hasattr(test_sparse, "A"))

print("GRN object:", adata_grn_log.shape)
print("TFs:", len(grn_tfs))

print("Formal loom:", GRN_LOOM.exists())
print("Smoke loom:", SMOKE_LOOM.exists())
print("Smoke TF list:", SMOKE_TF_TXT.exists())

assert scipy.__version__.startswith("1.13.")
assert hasattr(test_sparse, "A")
assert adata_grn_log.shape == (12057, 12916)
assert len(grn_tfs) == 1329
assert SMOKE_LOOM.exists()
assert SMOKE_TF_TXT.exists()

print("\nSTEP29E RECOVERY: PASS")

In [ ]:
# =============================================================================
# Step29E Cell 50
# Re-run GRNBoost2 smoke test after SciPy fix
# =============================================================================

import subprocess

# Remove the failed empty result from the previous incompatible run
if SMOKE_ADJ.exists():
    SMOKE_ADJ.unlink()

cmd_smoke = [
    sys.executable,
    "-m",
    "pyscenic.cli.arboreto_with_multiprocessing",

    str(SMOKE_LOOM),
    str(SMOKE_TF_TXT),

    "--method",
    "grnboost2",

    "--sparse",

    "--num_workers",
    "2",

    "--seed",
    "42",

    "--gene_attribute",
    "Gene",

    "--cell_id_attribute",
    "CellID",

    "-o",
    str(SMOKE_ADJ)
]

print("Running GRNBoost2 smoke test...")

with open(
    SMOKE_LOG,
    "w"
) as log:

    result = subprocess.run(
        cmd_smoke,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True
    )

print(
    "Return code:",
    result.returncode
)

print(
    "Output exists:",
    SMOKE_ADJ.exists()
)

print(
    "Log:",
    SMOKE_LOG
)

In [ ]:
# =============================================================================
# Step29E Cell 51
# Smoke-test validation
# =============================================================================

smoke_adj = pd.read_csv(
    SMOKE_ADJ,
    sep="\t"
)

print(
    "Shape:",
    smoke_adj.shape
)

print(
    "Columns:",
    smoke_adj.columns.tolist()
)

display(
    smoke_adj.head(20)
)

required_cols = {
    "TF",
    "target",
    "importance"
}

SMOKE_PASS = (
    result.returncode == 0
    and len(smoke_adj) > 0
    and required_cols.issubset(
        smoke_adj.columns
    )
    and np.isfinite(
        smoke_adj["importance"]
    ).all()
    and (
        smoke_adj["importance"] >= 0
    ).all()
)

print(
    "\nSMOKE_PASS:",
    SMOKE_PASS
)

assert SMOKE_PASS

print(
    "\nSTEP29E GRNBOOST2 SMOKE TEST: PASS"
)

In [ ]:
# =============================================================================
# Step29E Cell 52
# Formal environment audit
# =============================================================================

import json
import sklearn
import pyscenic
import platform

env_audit = {
    "Python": platform.python_version(),
    "Python_executable": sys.executable,
    "NumPy": np.__version__,
    "SciPy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "pySCENIC": pyscenic.__version__,
    "Cells": int(adata_grn_log.n_obs),
    "Genes": int(adata_grn_log.n_vars),
    "TFs": int(len(grn_tfs)),
    "Expression_transform": "log2(count + 1)",
    "GRN_method": "GRNBoost2",
    "Seed": 42
}

ENV_JSON = (
    AUDIT_DIR /
    "Step29E_GRNBoost2_Environment.json"
)

with open(
    ENV_JSON,
    "w"
) as f:
    json.dump(
        env_audit,
        f,
        indent=2
    )

display(
    pd.DataFrame(
        env_audit.items(),
        columns=[
            "Item",
            "Value"
        ]
    )
)

In [ ]:
# =============================================================================
# Step29E Cell 53
# Formal GRNBoost2 configuration lock
# =============================================================================

GRN_SEED = 42
GRN_WORKERS = 4

GRN_ADJ = (
    TABLE_DIR /
    "Step29E_GRNBoost2_Adjacencies.tsv"
)

GRN_LOG = (
    AUDIT_DIR /
    "Step29E_GRNBoost2_Run.log"
)

print(
    "Cells:",
    adata_grn_log.n_obs
)

print(
    "Genes:",
    adata_grn_log.n_vars
)

print(
    "TFs:",
    len(grn_tfs)
)

print(
    "Expression:",
    "log2(count + 1)"
)

print(
    "Workers:",
    GRN_WORKERS
)

print(
    "Seed:",
    GRN_SEED
)

print(
    "Adjacency output:",
    GRN_ADJ
)

print(
    "\nFORMAL CONFIG LOCKED"
)

In [ ]:
# =============================================================================
# Step29E Cell 54
# FORMAL GRNBoost2 inference
# =============================================================================

import os
import time
import subprocess

if GRN_ADJ.exists():

    raise FileExistsError(
        "Formal GRNBoost2 output already exists:\n"
        f"{GRN_ADJ}\n"
        "Existing formal results will not be overwritten."
    )


env = os.environ.copy()

# Prevent hidden thread multiplication inside each worker
env["OMP_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["NUMEXPR_NUM_THREADS"] = "1"


cmd_grn = [
    sys.executable,
    "-m",
    "pyscenic.cli.arboreto_with_multiprocessing",

    str(GRN_LOOM),
    str(GRN_TF_TXT),

    "--method",
    "grnboost2",

    "--sparse",

    "--num_workers",
    str(GRN_WORKERS),

    "--seed",
    str(GRN_SEED),

    "--gene_attribute",
    "Gene",

    "--cell_id_attribute",
    "CellID",

    "-o",
    str(GRN_ADJ)
]


print(
    "Starting formal GRNBoost2..."
)

print(
    "\nCommand:"
)

print(
    " ".join(cmd_grn)
)

start_time = time.time()


with open(
    GRN_LOG,
    "w"
) as log:

    process = subprocess.Popen(
        cmd_grn,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    for line in process.stdout:

        print(
            line,
            end=""
        )

        log.write(
            line
        )

        log.flush()

    return_code = (
        process.wait()
    )


elapsed_hours = (
    time.time()
    - start_time
) / 3600


print(
    "\nReturn code:",
    return_code
)

print(
    "Elapsed hours:",
    round(
        elapsed_hours,
        3
    )
)

print(
    "Output exists:",
    GRN_ADJ.exists()
)

print(
    "Run log:",
    GRN_LOG
)


if return_code != 0:

    log_lines = (
        GRN_LOG
        .read_text(
            errors="ignore"
        )
        .splitlines()
    )

    print(
        "\nLAST 100 LOG LINES:"
    )

    print(
        "\n".join(
            log_lines[-100:]
        )
    )


assert return_code == 0
assert GRN_ADJ.exists()

print(
    "\nFORMAL GRNBOOST2: COMPLETE"
)

In [ ]:
# =============================================================================
# Step29E Cell 55
# Formal GRNBoost2 adjacency QC
# =============================================================================

adj = pd.read_csv(
    GRN_ADJ,
    sep="\t"
)

print(
    "Adjacency shape:",
    adj.shape
)

print(
    "Columns:",
    adj.columns.tolist()
)

display(
    adj.head(20)
)


required_cols = {
    "TF",
    "target",
    "importance"
}

assert required_cols.issubset(
    adj.columns
)

assert len(adj) > 0


adj["TF"] = (
    adj["TF"]
    .astype(str)
)

adj["target"] = (
    adj["target"]
    .astype(str)
)


all_grn_genes = set(
    adata_grn_log
    .var_names
    .astype(str)
)

all_grn_tfs = set(
    grn_tfs
)


qc = {
    "N_edges":
        int(len(adj)),

    "N_TFs_with_edges":
        int(
            adj["TF"].nunique()
        ),

    "N_targets":
        int(
            adj["target"].nunique()
        ),

    "All_TFs_valid":
        set(
            adj["TF"]
        ).issubset(
            all_grn_tfs
        ),

    "All_targets_valid":
        set(
            adj["target"]
        ).issubset(
            all_grn_genes
        ),

    "N_nonfinite_importance":
        int(
            (
                ~np.isfinite(
                    adj["importance"]
                )
            ).sum()
        ),

    "N_negative_importance":
        int(
            (
                adj["importance"] < 0
            ).sum()
        ),

    "N_self_edges":
        int(
            (
                adj["TF"]
                ==
                adj["target"]
            ).sum()
        )
}

qc_df = pd.DataFrame(
    [qc]
)

display(
    qc_df
)

In [ ]:
# =============================================================================
# Step29E Cell 56
# TF-level network summary
# =============================================================================

tf_summary = (
    adj
    .groupby(
        "TF"
    )
    .agg(
        N_targets=(
            "target",
            "nunique"
        ),

        Total_importance=(
            "importance",
            "sum"
        ),

        Mean_importance=(
            "importance",
            "mean"
        ),

        Max_importance=(
            "importance",
            "max"
        )
    )
    .sort_values(
        "Total_importance",
        ascending=False
    )
)

display(
    tf_summary.head(30)
)

tf_summary.to_csv(
    TABLE_DIR /
    "Step29E_GRNBoost2_TF_Summary.csv"
)

In [ ]:
# =============================================================================
# Step29E Cell 57
# Save GRNBoost2 QC
# =============================================================================

qc_df.to_csv(
    TABLE_DIR /
    "Step29E_GRNBoost2_QC.csv",
    index=False
)

print(
    "QC saved."
)

In [ ]:
# =============================================================================
# Step29E Cell 58
# REPORT — GRNBoost2 network inference
# =============================================================================

GRNBOOST_PASS = (
    len(adj) > 0
    and qc["All_TFs_valid"]
    and qc["All_targets_valid"]
    and qc[
        "N_nonfinite_importance"
    ] == 0
    and qc[
        "N_negative_importance"
    ] == 0
)

print("=" * 100)

print(
    "STEP29E GRNBOOST2 REPORT"
)

print("=" * 100)


print(
    "\nExpression source:"
)

print(
    "Frozen Step29D filtered raw-count object"
)

print(
    "GRN input transform:"
)

print(
    "log2(count + 1)"
)


print(
    "\nCells:",
    adata_grn_log.n_obs
)

print(
    "Genes:",
    adata_grn_log.n_vars
)

print(
    "TFs supplied:",
    len(grn_tfs)
)


print(
    "\nInferred edges:",
    qc["N_edges"]
)

print(
    "TFs with edges:",
    qc[
        "N_TFs_with_edges"
    ]
)

print(
    "Targets represented:",
    qc[
        "N_targets"
    ]
)

print(
    "Self edges:",
    qc[
        "N_self_edges"
    ]
)


print(
    "\nSeed:",
    GRN_SEED
)

print(
    "Workers:",
    GRN_WORKERS
)


print(
    "\nTop 20 TFs by total GRNBoost2 importance:"
)

display(
    tf_summary.head(20)
)


print(
    "\nGRNBOOST_PASS:",
    GRNBOOST_PASS
)


print(
    "\nInterpretation boundary:"
)

print(
    "GRNBoost2 defines candidate TF-target "
    "co-expression dependencies only."
)

print(
    "These edges are not yet motif-supported regulons."
)

print(
    "State labels, Core50, pseudotime, and biological "
    "programs were not used during GRN inference."
)

print(
    "\nSTEP29E COMPLETE"
)

In [ ]:
# =============================================================================
# Step29F Cell 59
# Lock motif-pruning resources
# =============================================================================

from pathlib import Path
import sys

RESOURCE_DIR = (
    STEP29_DIR /
    "Resources"
)

TF_FILE = (
    RESOURCE_DIR /
    "tf" /
    "allTFs_hg38.txt"
)

MOTIF_FILE = (
    RESOURCE_DIR /
    "motif" /
    "motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl"
)

DB_10KB = (
    RESOURCE_DIR /
    "cistarget" /
    "hg38_10kbp_up_10kbp_down_full_tx_v10_clust."
    "genes_vs_motifs.rankings.feather"
)

DB_500BP = (
    RESOURCE_DIR /
    "cistarget" /
    "hg38_500bp_up_100bp_down_full_tx_v10_clust."
    "genes_vs_motifs.rankings.feather"
)

PYSCENIC_CLI = (
    Path(sys.executable).parent /
    "pyscenic"
)

files29f = {
    "Adjacency": GRN_ADJ,
    "Expression_loom": GRN_LOOM,
    "Motif_annotation": MOTIF_FILE,
    "Ranking_10kb": DB_10KB,
    "Ranking_500bp": DB_500BP,
    "pySCENIC_CLI": PYSCENIC_CLI
}

for name, path in files29f.items():

    print(
        name,
        "->",
        path.exists()
    )

assert all(
    p.exists()
    for p in files29f.values()
)

print(
    "\nSTEP29F RESOURCE LOCK: PASS"
)

In [ ]:
# =============================================================================
# Step29F Cell 60
# Formal motif-pruning configuration
# =============================================================================

CTX_WORKERS = 2
CTX_CHUNK_SIZE = 50

RANK_THRESHOLD = 5000
AUC_THRESHOLD = 0.05
NES_THRESHOLD = 3.0

MOTIFS_CSV = (
    TABLE_DIR /
    "Step29F_MotifPruned_Regulons.csv"
)

MOTIFS_TMP = (
    TABLE_DIR /
    "Step29F_MotifPruned_Regulons.partial.csv"
)

CTX_LOG = (
    AUDIT_DIR /
    "Step29F_CTX_Run.log"
)

print(
    "Adjacency:",
    GRN_ADJ
)

print(
    "Ranking DBs:"
)

print(
    DB_10KB
)

print(
    DB_500BP
)

print(
    "Motif annotations:",
    MOTIF_FILE
)

print(
    "\nWorkers:",
    CTX_WORKERS
)

print(
    "Chunk size:",
    CTX_CHUNK_SIZE
)

print(
    "Rank threshold:",
    RANK_THRESHOLD
)

print(
    "AUC threshold:",
    AUC_THRESHOLD
)

print(
    "NES threshold:",
    NES_THRESHOLD
)

print(
    "\nPrimary analysis:"
)

print(
    "Positive TF-target modules only"
)

print(
    "mask_dropouts = False"
)

print(
    "\nSTEP29F CONFIG LOCKED"
)

In [ ]:
# =============================================================================
# Step29F Cell 61
# ctx CLI preflight
# =============================================================================

import subprocess

res = subprocess.run(
    [
        str(PYSCENIC_CLI),
        "ctx",
        "-h"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(
    "Return code:",
    res.returncode
)

print(
    res.stdout[:4000]
)

if res.stderr:

    print(
        "\nSTDERR:"
    )

    print(
        res.stderr[:2000]
    )

assert res.returncode == 0

print(
    "\nPYSCENIC CTX PREFLIGHT: PASS"
)

In [ ]:
# =============================================================================
# Step29F Cell 62
# FORMAL motif pruning with cisTarget
# =============================================================================

import os
import time
import subprocess

if MOTIFS_CSV.exists():

    raise FileExistsError(
        "Final Step29F motif output already exists:\n"
        f"{MOTIFS_CSV}\n"
        "Existing formal results will not be overwritten."
    )


if MOTIFS_TMP.exists():

    MOTIFS_TMP.unlink()


env = os.environ.copy()

env["OMP_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["NUMEXPR_NUM_THREADS"] = "1"


cmd_ctx = [
    str(PYSCENIC_CLI),

    "ctx",

    str(GRN_ADJ),

    str(DB_10KB),
    str(DB_500BP),

    "--annotations_fname",
    str(MOTIF_FILE),

    "--expression_mtx_fname",
    str(GRN_LOOM),

    "--mode",
    "custom_multiprocessing",

    "--num_workers",
    str(CTX_WORKERS),

    "--chunk_size",
    str(CTX_CHUNK_SIZE),

    "--rank_threshold",
    str(RANK_THRESHOLD),

    "--auc_threshold",
    str(AUC_THRESHOLD),

    "--nes_threshold",
    str(NES_THRESHOLD),

    "--thresholds",
    "0.75",
    "0.90",

    "--top_n_targets",
    "50",

    "--top_n_regulators",
    "5",
    "10",
    "50",

    "--min_genes",
    "20",

    "--gene_attribute",
    "Gene",

    "--cell_id_attribute",
    "CellID",

    "--output",
    str(MOTIFS_TMP)
]


print(
    "Starting formal motif pruning..."
)

print(
    "\nCommand:"
)

print(
    " ".join(cmd_ctx)
)


start_time = time.time()


with open(
    CTX_LOG,
    "w"
) as log:

    process = subprocess.Popen(
        cmd_ctx,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    for line in process.stdout:

        print(
            line,
            end=""
        )

        log.write(
            line
        )

        log.flush()

    ctx_return_code = (
        process.wait()
    )


ctx_elapsed_hours = (
    time.time()
    - start_time
) / 3600


print(
    "\nReturn code:",
    ctx_return_code
)

print(
    "Elapsed hours:",
    round(
        ctx_elapsed_hours,
        3
    )
)

print(
    "Partial output exists:",
    MOTIFS_TMP.exists()
)

if MOTIFS_TMP.exists():

    print(
        "Partial output MB:",
        round(
            MOTIFS_TMP.stat().st_size
            / 1024**2,
            2
        )
    )


if ctx_return_code != 0:

    log_lines = (
        CTX_LOG
        .read_text(
            errors="ignore"
        )
        .splitlines()
    )

    print(
        "\nLAST 100 LOG LINES:"
    )

    print(
        "\n".join(
            log_lines[-100:]
        )
    )


assert ctx_return_code == 0
assert MOTIFS_TMP.exists()
assert MOTIFS_TMP.stat().st_size > 0

MOTIFS_TMP.replace(
    MOTIFS_CSV
)

print(
    "\nFORMAL MOTIF PRUNING: COMPLETE"
)

print(
    MOTIFS_CSV
)

In [ ]:
# =============================================================================
# Step29F Cell 63
# Load motif-pruned result
# =============================================================================

from pyscenic.utils import load_motifs
from pyscenic.transform import df2regulons

motifs_df = load_motifs(
    str(MOTIFS_CSV)
)

print(
    "Motif table shape:",
    motifs_df.shape
)

print(
    "Index names:",
    motifs_df.index.names
)

print(
    "Column levels:",
    motifs_df.columns.nlevels
)

print(
    "\nFirst motif rows:"
)

display(
    motifs_df.head()
)

assert not motifs_df.empty

print(
    "\nMOTIF TABLE LOAD: PASS"
)

In [ ]:
# =============================================================================
# Step29F Cell 64
# Convert motif-pruned table to regulons
# =============================================================================

regulons = df2regulons(
    motifs_df
)

print(
    "Number of regulons:",
    len(regulons)
)

assert len(regulons) > 0


regulon_rows = []

for r in regulons:

    tf_name = getattr(
        r,
        "transcription_factor",
        r.name.split("(")[0]
    )

    regulon_rows.append({
        "Regulon":
            r.name,

        "TF":
            tf_name,

        "N_targets":
            len(
                r.gene2weight
            )
    })


regulon_summary = (
    pd.DataFrame(
        regulon_rows
    )
    .sort_values(
        "N_targets",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

display(
    regulon_summary.head(30)
)

In [ ]:
# =============================================================================
# Step29F Cell 65
# Sentinel regulator motif-support audit
# =============================================================================

sentinel_tfs = [
    "HNF4A",
    "HNF1A",
    "FOXM1",
    "E2F1",
    "E2F8",
    "MYBL2",
    "MYC",
    "STAT3",
    "SMAD3"
]

sentinel_regulon_audit = []

for tf in sentinel_tfs:

    sub = regulon_summary.loc[
        regulon_summary[
            "TF"
        ] == tf
    ]

    sentinel_regulon_audit.append({
        "TF":
            tf,

        "Motif_supported":
            len(sub) > 0,

        "N_regulons":
            len(sub),

        "Max_targets":
            int(
                sub[
                    "N_targets"
                ].max()
            )
            if len(sub) > 0
            else 0
    })


sentinel_regulon_audit = pd.DataFrame(
    sentinel_regulon_audit
)

display(
    sentinel_regulon_audit
)

In [ ]:
# =============================================================================
# Step29F Cell 66
# Save regulon objects and summaries
# =============================================================================

import pickle
import json

REGULONS_DAT = (
    DATA_DIR /
    "Step29F_Regulons.dat"
)

REGULON_SUMMARY_CSV = (
    TABLE_DIR /
    "Step29F_Regulon_Summary.csv"
)

REGULON_TARGETS_JSON = (
    DATA_DIR /
    "Step29F_Regulon_Targets.json"
)


with open(
    REGULONS_DAT,
    "wb"
) as f:

    pickle.dump(
        regulons,
        f
    )


regulon_summary.to_csv(
    REGULON_SUMMARY_CSV,
    index=False
)


regulon_targets = {
    r.name:
        list(
            r.gene2weight.keys()
        )
    for r in regulons
}


with open(
    REGULON_TARGETS_JSON,
    "w"
) as f:

    json.dump(
        regulon_targets,
        f,
        indent=2
    )


sentinel_regulon_audit.to_csv(
    TABLE_DIR /
    "Step29F_Sentinel_Regulon_Audit.csv",
    index=False
)

print(
    "Saved:",
    REGULONS_DAT
)

print(
    "Saved:",
    REGULON_SUMMARY_CSV
)

print(
    "Saved:",
    REGULON_TARGETS_JSON
)

In [ ]:
# =============================================================================
# Step29F Cell 67
# REPORT — motif-supported regulon definition
# =============================================================================

motif_tf_count = (
    motifs_df
    .index
    .get_level_values(
        0
    )
    .nunique()
)

motif_id_count = (
    motifs_df
    .index
    .get_level_values(
        1
    )
    .nunique()
)

REGULON_PASS = (
    not motifs_df.empty
    and
    len(regulons) > 0
    and
    regulon_summary[
        "N_targets"
    ].min() > 0
)

print("=" * 100)

print(
    "STEP29F MOTIF PRUNING REPORT"
)

print("=" * 100)


print(
    "\nInput GRN edges:",
    len(adj)
)

print(
    "Input TFs:",
    adj["TF"].nunique()
)


print(
    "\nMotif-enrichment rows:",
    len(motifs_df)
)

print(
    "TFs with motif-supported evidence:",
    motif_tf_count
)

print(
    "Unique enriched motifs:",
    motif_id_count
)


print(
    "\nFinal regulons:",
    len(regulons)
)

print(
    "Median targets per regulon:",
    float(
        regulon_summary[
            "N_targets"
        ].median()
    )
)

print(
    "Maximum targets per regulon:",
    int(
        regulon_summary[
            "N_targets"
        ].max()
    )
)


print(
    "\nLargest regulons:"
)

display(
    regulon_summary.head(20)
)


print(
    "\nSentinel regulators:"
)

display(
    sentinel_regulon_audit
)


print(
    "\nREGULON_PASS:",
    REGULON_PASS
)


print(
    "\nInterpretation boundary:"
)

print(
    "These regulons are supported by "
    "co-expression plus cis-regulatory motif enrichment."
)

print(
    "They are not yet evidence of state-specific regulatory activity."
)

print(
    "MS1-MS8 labels, Core50, pseudotime, and program scores "
    "have still not been used for regulon selection."
)

print(
    "\nSTEP29F COMPLETE"
)

In [ ]:
# =============================================================================
# Step29G Cell 68
# FORMAL AUCell regulon activity inference
# =============================================================================

import os
import time
import subprocess
from pathlib import Path

AUC_WORKERS = 2
AUC_SEED = 42
AUC_THRESHOLD = 0.05

AUC_CSV = (
    TABLE_DIR /
    "Step29G_AUCell_RegulonActivity.csv"
)

AUC_TMP = (
    TABLE_DIR /
    "Step29G_AUCell_RegulonActivity.partial.csv"
)

AUC_LOG = (
    AUDIT_DIR /
    "Step29G_AUCell_Run.log"
)

# ctx output
REGULON_SIGNATURES = MOTIFS_CSV

assert GRN_LOOM.exists()
assert REGULON_SIGNATURES.exists()

if AUC_CSV.exists():
    raise FileExistsError(
        f"Final AUCell output already exists:\n{AUC_CSV}"
    )

if AUC_TMP.exists():
    AUC_TMP.unlink()


env = os.environ.copy()

env["OMP_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["NUMEXPR_NUM_THREADS"] = "1"


cmd_auc = [
    str(PYSCENIC_CLI),

    "aucell",

    str(GRN_LOOM),
    str(REGULON_SIGNATURES),

    "--output",
    str(AUC_TMP),

    "--num_workers",
    str(AUC_WORKERS),

    "--seed",
    str(AUC_SEED),

    "--auc_threshold",
    str(AUC_THRESHOLD),

    "--gene_attribute",
    "Gene",

    "--cell_id_attribute",
    "CellID"
]


print("Starting AUCell...")
print("\nCommand:")
print(" ".join(cmd_auc))

start_time = time.time()

with open(
    AUC_LOG,
    "w"
) as log:

    process = subprocess.Popen(
        cmd_auc,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    for line in process.stdout:

        print(
            line,
            end=""
        )

        log.write(line)
        log.flush()

    auc_return_code = process.wait()


auc_elapsed_minutes = (
    time.time()
    - start_time
) / 60


print(
    "\nReturn code:",
    auc_return_code
)

print(
    "Elapsed minutes:",
    round(
        auc_elapsed_minutes,
        2
    )
)

print(
    "Output exists:",
    AUC_TMP.exists()
)


if auc_return_code != 0:

    lines = (
        AUC_LOG
        .read_text(
            errors="ignore"
        )
        .splitlines()
    )

    print("\nLAST 80 LOG LINES:")
    print(
        "\n".join(
            lines[-80:]
        )
    )


assert auc_return_code == 0
assert AUC_TMP.exists()
assert AUC_TMP.stat().st_size > 0

AUC_TMP.replace(
    AUC_CSV
)

print(
    "\nSTEP29G AUCELL CALCULATION: COMPLETE"
)

In [ ]:
# =============================================================================
# Step29G Cell 69
# AUCell activity matrix QC
# =============================================================================

auc_mtx = pd.read_csv(
    AUC_CSV,
    index_col=0
)

print(
    "AUC matrix shape:",
    auc_mtx.shape
)

print(
    "Expected cells:",
    12057
)

print(
    "Expected regulons:",
    len(regulons)
)

print(
    "\nFirst cells:"
)

display(
    auc_mtx.iloc[:5, :10]
)


# -------------------------------------------------------------------------
# Basic QC
# -------------------------------------------------------------------------

same_cell_set = (
    set(
        auc_mtx.index.astype(str)
    )
    ==
    set(
        adata_grn_log.obs_names.astype(str)
    )
)

n_nonfinite = int(
    (~np.isfinite(
        auc_mtx.to_numpy()
    )).sum()
)

n_negative = int(
    (
        auc_mtx.to_numpy() < 0
    ).sum()
)

n_constant_regulons = int(
    (
        auc_mtx.nunique(
            axis=0
        ) <= 1
    ).sum()
)

print(
    "\nSame cell set:",
    same_cell_set
)

print(
    "Non-finite values:",
    n_nonfinite
)

print(
    "Negative AUC values:",
    n_negative
)

print(
    "Constant regulons:",
    n_constant_regulons
)


assert auc_mtx.shape[0] == 12057
assert same_cell_set
assert n_nonfinite == 0
assert n_negative == 0

print(
    "\nAUC MATRIX QC: PASS"
)

In [ ]:
# =============================================================================
# Step29G Cell 70
# REPORT — single-cell regulon activity
# =============================================================================

auc_summary = pd.DataFrame({
    "Regulon":
        auc_mtx.columns,

    "Mean_AUC":
        auc_mtx.mean(
            axis=0
        ).values,

    "Median_AUC":
        auc_mtx.median(
            axis=0
        ).values,

    "SD_AUC":
        auc_mtx.std(
            axis=0
        ).values,

    "Min_AUC":
        auc_mtx.min(
            axis=0
        ).values,

    "Max_AUC":
        auc_mtx.max(
            axis=0
        ).values
})

auc_summary = (
    auc_summary
    .sort_values(
        "SD_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

auc_summary.to_csv(
    TABLE_DIR /
    "Step29G_RegulonActivity_Summary.csv",
    index=False
)


AUC_PASS = (
    auc_mtx.shape[0] == 12057
    and
    auc_mtx.shape[1] > 0
    and
    same_cell_set
    and
    n_nonfinite == 0
    and
    n_negative == 0
)


print("=" * 100)

print(
    "STEP29G AUCELL REPORT"
)

print("=" * 100)

print(
    "\nCells:",
    auc_mtx.shape[0]
)

print(
    "Regulons scored:",
    auc_mtx.shape[1]
)

print(
    "Non-finite:",
    n_nonfinite
)

print(
    "Negative values:",
    n_negative
)

print(
    "Constant regulons:",
    n_constant_regulons
)


print(
    "\nMost variable regulons:"
)

display(
    auc_summary.head(20)
)


print(
    "\nAUC_PASS:",
    AUC_PASS
)


print(
    "\nInterpretation boundary:"
)

print(
    "AUCell provides cell-level regulon activity scores."
)

print(
    "No MS-state, pseudotime, Core50, or program information "
    "was used to calculate these activities."
)

print(
    "\nSTEP29G COMPLETE"
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step29H Cell 71
# Align AUCell activity with frozen biological annotations
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as stats

ROOT = Path(_hcc_path(''))

STEP29_DIR = ROOT / "LIHC_scRNA_Step29_GRN"
DATA_DIR = STEP29_DIR / "Data"
TABLE_DIR = STEP29_DIR / "Tables"
AUDIT_DIR = STEP29_DIR / "Audit"

STEP27_H5AD = (
    ROOT /
    "LIHC_scRNA_Step27_Trajectory" /
    "Data" /
    "GSE149614_STEP27_FINAL_TRAJECTORY.h5ad"
)

AUC_CSV = (
    TABLE_DIR /
    "Step29G_AUCell_RegulonActivity.csv"
)

REGULON_TARGETS_JSON = (
    DATA_DIR /
    "Step29F_Regulon_Targets.json"
)

adata29 = sc.read_h5ad(
    STEP27_H5AD
)

auc_mtx = pd.read_csv(
    AUC_CSV,
    index_col=0
)

auc_mtx.index = (
    auc_mtx.index
    .astype(str)
)

# Reorder exactly to frozen Step27 cell order
auc_mtx = auc_mtx.loc[
    adata29.obs_names.astype(str)
]

STATE_COL = "final_malignant_state_id"
PATIENT_COL = "patient"
PT_COL = "MS6_anchored_pseudotime"

CORE_COL = "MS3_conserved_Core50_score"
G2M_COL = "Program__Proliferation_G2M"
MITOTIC_COL = "DeepProgram__Mitotic_Entry"
SPINDLE_COL = "DeepProgram__Spindle_Chromosome_Segregation"
HEPATIC_COL = "DeepProgram__Hepatic_Differentiation"
BILE_COL = "DeepProgram__Bile_Acid_Metabolism"

meta_cols = [
    PATIENT_COL,
    STATE_COL,
    PT_COL,
    CORE_COL,
    G2M_COL,
    MITOTIC_COL,
    SPINDLE_COL,
    HEPATIC_COL,
    BILE_COL
]

meta29 = (
    adata29.obs[
        meta_cols
    ]
    .copy()
)

assert np.array_equal(
    auc_mtx.index,
    meta29.index.astype(str)
)

print(
    "AUCell:",
    auc_mtx.shape
)

print(
    "Metadata:",
    meta29.shape
)

print(
    "\nSTEP29H ALIGNMENT: PASS"
)

In [ ]:
# =============================================================================
# Step29H Cell 72 — FIXED
# Primary MS6-MS5-MS3 axis + zero-variance regulon audit
# =============================================================================

AXIS_STATES = [
    "MS6",
    "MS5",
    "MS3"
]

AXIS_PATIENTS = [
    "HCC09",
    "HCC10"
]

# -------------------------------------------------------------------------
# 1. Select primary biological axis
# -------------------------------------------------------------------------

axis_mask = (
    adata29.obs[
        "final_malignant_state_id"
    ].isin(AXIS_STATES)
    &
    adata29.obs[
        "patient"
    ].isin(AXIS_PATIENTS)
)

axis_cells = (
    adata29.obs_names[
        axis_mask
    ]
    .astype(str)
)

axis_meta = (
    adata29.obs.loc[
        axis_cells
    ]
    .copy()
)

# Align AUCell matrix exactly to axis cells
axis_auc = (
    auc_mtx
    .loc[
        axis_cells
    ]
    .copy()
)

print(
    "Axis cells:",
    axis_auc.shape[0]
)

print(
    "Regulons:",
    axis_auc.shape[1]
)

print(
    "\nPatient × state counts:"
)

display(
    pd.crosstab(
        axis_meta["patient"],
        axis_meta[
            "final_malignant_state_id"
        ]
    )
)

In [ ]:
# =============================================================================
# Step29H Cell 72B
# Axis-specific regulon variance audit
# =============================================================================

axis_reg_mean = (
    axis_auc.mean(
        axis=0
    )
)

axis_reg_sd = (
    axis_auc.std(
        axis=0,
        ddof=0
    )
)

variance_audit = pd.DataFrame({
    "Regulon":
        axis_auc.columns.astype(str),

    "Axis_mean":
        axis_reg_mean.values,

    "Axis_SD":
        axis_reg_sd.values
})

variance_audit[
    "Finite_SD"
] = np.isfinite(
    variance_audit[
        "Axis_SD"
    ]
)

variance_audit[
    "Variable_on_axis"
] = (
    variance_audit[
        "Finite_SD"
    ]
    &
    (
        variance_audit[
            "Axis_SD"
        ] > 1e-12
    )
)

constant_axis_regulons = (
    variance_audit.loc[
        ~variance_audit[
            "Variable_on_axis"
        ],
        "Regulon"
    ]
    .tolist()
)

variable_axis_regulons = (
    variance_audit.loc[
        variance_audit[
            "Variable_on_axis"
        ],
        "Regulon"
    ]
    .tolist()
)

print(
    "Total regulons:",
    len(
        variance_audit
    )
)

print(
    "Variable on axis:",
    len(
        variable_axis_regulons
    )
)

print(
    "Zero/non-variable on axis:",
    len(
        constant_axis_regulons
    )
)

if constant_axis_regulons:

    print(
        "\nAxis-nonvariable regulons:"
    )

    print(
        constant_axis_regulons
    )

variance_audit.to_csv(
    TABLE_DIR /
    "Step29H_Axis_Regulon_Variance_Audit.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step29H Cell 72C
# Axis-specific regulon z-score matrix
# =============================================================================

# Only regulons with variance on the primary axis
axis_auc_use = (
    axis_auc[
        variable_axis_regulons
    ]
    .copy()
)

axis_mean_use = (
    axis_auc_use.mean(
        axis=0
    )
)

axis_sd_use = (
    axis_auc_use.std(
        axis=0,
        ddof=0
    )
)

axis_auc_z = (
    axis_auc_use
    - axis_mean_use
) / axis_sd_use

n_nan = int(
    axis_auc_z
    .isna()
    .sum()
    .sum()
)

n_nonfinite = int(
    (
        ~np.isfinite(
            axis_auc_z.to_numpy()
        )
    ).sum()
)

print(
    "Axis cells:",
    axis_auc_z.shape[0]
)

print(
    "Regulons retained for axis analysis:",
    axis_auc_z.shape[1]
)

print(
    "Regulons excluded only because "
    "of zero axis variance:",
    len(
        constant_axis_regulons
    )
)

print(
    "NaN:",
    n_nan
)

print(
    "Non-finite:",
    n_nonfinite
)

assert axis_auc_z.shape[0] == 3280
assert n_nan == 0
assert n_nonfinite == 0

print(
    "\nSTEP29H PRIMARY AXIS: PASS"
)

In [ ]:
# =============================================================================
# Step29H Cell 73
# Lock biological validation features
# =============================================================================

STATE_COL = "final_malignant_state_id"
PATIENT_COL = "patient"
PT_COL = "MS6_anchored_pseudotime"

feature_map = {
    "PT":
        PT_COL,

    "Core50":
        "MS3_conserved_Core50_score",

    "G2M":
        "Program__Proliferation_G2M",

    "Mitotic":
        "DeepProgram__Mitotic_Entry",

    "Spindle":
        "DeepProgram__Spindle_Chromosome_Segregation",

    "Hepatic":
        "DeepProgram__Hepatic_Differentiation",

    "BileAcid":
        "DeepProgram__Bile_Acid_Metabolism"
}

axis_meta = (
    adata29.obs
    .loc[
        axis_auc_use.index
    ]
    .copy()
)

missing_features = [
    col
    for col in feature_map.values()
    if col not in axis_meta.columns
]

print(
    "Missing features:",
    missing_features
)

assert len(missing_features) == 0

print(
    "Axis cells:",
    axis_meta.shape[0]
)

print(
    "Axis regulons:",
    axis_auc_use.shape[1]
)

print(
    "\nSTEP29H FEATURE LOCK: PASS"
)

In [ ]:
# =============================================================================
# Step29H Cell 74
# Patient-specific regulon-biological correlations
# =============================================================================

def safe_spearman(x, y):

    tmp = pd.DataFrame({
        "x": x,
        "y": y
    }).dropna()

    if len(tmp) < 20:
        return np.nan

    if (
        tmp["x"].nunique() < 2
        or
        tmp["y"].nunique() < 2
    ):
        return np.nan

    return float(
        tmp["x"].corr(
            tmp["y"],
            method="spearman"
        )
    )


correlation_rows = []

for patient in AXIS_PATIENTS:

    patient_cells = (
        axis_meta.index[
            axis_meta[
                PATIENT_COL
            ] == patient
        ]
    )

    auc_p = (
        axis_auc_use
        .loc[
            patient_cells
        ]
    )

    meta_p = (
        axis_meta
        .loc[
            patient_cells
        ]
    )

    for regulon in auc_p.columns:

        row = {
            "Regulon":
                regulon,

            "Patient":
                patient,

            "N_cells":
                len(patient_cells)
        }

        for short_name, col in feature_map.items():

            row[
                f"rho_{short_name}"
            ] = safe_spearman(
                auc_p[regulon],
                meta_p[col]
            )

        correlation_rows.append(
            row
        )


corr_long = pd.DataFrame(
    correlation_rows
)

print(
    "Correlation rows:",
    corr_long.shape
)

display(
    corr_long.head()
)

In [ ]:
# =============================================================================
# Step29H Cell 75
# Cross-patient correlation matrix
# =============================================================================

corr_wide_parts = []

for short_name in feature_map.keys():

    tmp = (
        corr_long
        .pivot(
            index="Regulon",
            columns="Patient",
            values=f"rho_{short_name}"
        )
    )

    tmp.columns = [
        f"rho_{short_name}_{c}"
        for c in tmp.columns
    ]

    corr_wide_parts.append(
        tmp
    )


corr_wide = pd.concat(
    corr_wide_parts,
    axis=1
)

corr_wide = (
    corr_wide
    .reindex(
        axis_auc_use.columns
    )
)

print(
    "Correlation matrix:",
    corr_wide.shape
)

display(
    corr_wide.head()
)


In [ ]:
# =============================================================================
# Step29H Cell 76
# Patient-standardized state effects
# =============================================================================

state_effect_rows = []

for patient in AXIS_PATIENTS:

    patient_cells = (
        axis_meta.index[
            axis_meta[
                PATIENT_COL
            ] == patient
        ]
    )

    auc_p = (
        axis_auc_use
        .loc[
            patient_cells
        ]
        .copy()
    )

    state_p = (
        axis_meta
        .loc[
            patient_cells,
            STATE_COL
        ]
    )

    for regulon in auc_p.columns:

        x = (
            auc_p[
                regulon
            ]
            .astype(float)
        )

        sd = float(
            x.std(
                ddof=0
            )
        )

        if (
            not np.isfinite(sd)
            or
            sd <= 1e-12
        ):

            z = pd.Series(
                np.nan,
                index=x.index
            )

        else:

            z = (
                x - x.mean()
            ) / sd

        medians = {}

        for state in AXIS_STATES:

            vals = (
                z.loc[
                    state_p == state
                ]
            )

            medians[state] = (
                float(
                    vals.median()
                )
                if vals.notna().any()
                else np.nan
            )

        delta_36 = (
            medians["MS3"]
            -
            medians["MS6"]
        )

        transition_margin = (
            medians["MS5"]
            -
            max(
                medians["MS6"],
                medians["MS3"]
            )
            if np.all(
                np.isfinite([
                    medians["MS6"],
                    medians["MS5"],
                    medians["MS3"]
                ])
            )
            else np.nan
        )

        state_effect_rows.append({
            "Regulon":
                regulon,

            "Patient":
                patient,

            "MS6_median_z":
                medians["MS6"],

            "MS5_median_z":
                medians["MS5"],

            "MS3_median_z":
                medians["MS3"],

            "Delta_MS3_MS6":
                delta_36,

            "MS5_peak_margin":
                transition_margin
        })


state_effect_long = pd.DataFrame(
    state_effect_rows
)

display(
    state_effect_long.head()
)

In [ ]:
# =============================================================================
# Step29H Cell 77
# Cross-patient state-effect summary
# =============================================================================

metrics = [
    "MS6_median_z",
    "MS5_median_z",
    "MS3_median_z",
    "Delta_MS3_MS6",
    "MS5_peak_margin"
]

state_wide_parts = []

for metric in metrics:

    tmp = (
        state_effect_long
        .pivot(
            index="Regulon",
            columns="Patient",
            values=metric
        )
    )

    tmp.columns = [
        f"{metric}_{c}"
        for c in tmp.columns
    ]

    state_wide_parts.append(
        tmp
    )


state_wide = pd.concat(
    state_wide_parts,
    axis=1
)

state_wide = (
    state_wide
    .reindex(
        axis_auc_use.columns
    )
)


state_wide[
    "Mean_Delta_MS3_MS6"
] = state_wide[
    [
        "Delta_MS3_MS6_HCC09",
        "Delta_MS3_MS6_HCC10"
    ]
].mean(
    axis=1
)


state_wide[
    "Mean_MS5_peak_margin"
] = state_wide[
    [
        "MS5_peak_margin_HCC09",
        "MS5_peak_margin_HCC10"
    ]
].mean(
    axis=1
)


state_wide[
    "Delta_positive_both"
] = (
    (
        state_wide[
            "Delta_MS3_MS6_HCC09"
        ] > 0
    )
    &
    (
        state_wide[
            "Delta_MS3_MS6_HCC10"
        ] > 0
    )
)


state_wide[
    "Delta_negative_both"
] = (
    (
        state_wide[
            "Delta_MS3_MS6_HCC09"
        ] < 0
    )
    &
    (
        state_wide[
            "Delta_MS3_MS6_HCC10"
        ] < 0
    )
)


state_wide[
    "MS5_peak_both"
] = (
    (
        state_wide[
            "MS5_peak_margin_HCC09"
        ] > 0
    )
    &
    (
        state_wide[
            "MS5_peak_margin_HCC10"
        ] > 0
    )
)

print(
    "State summary:",
    state_wide.shape
)

print(
    "MS3>MS6 in both patients:",
    int(
        state_wide[
            "Delta_positive_both"
        ].sum()
    )
)

print(
    "MS6>MS3 in both patients:",
    int(
        state_wide[
            "Delta_negative_both"
        ].sum()
    )
)

print(
    "MS5 peak in both patients:",
    int(
        state_wide[
            "MS5_peak_both"
        ].sum()
    )
)

In [ ]:
# =============================================================================
# Step29H Cell 78
# Build integrated regulatory landscape
# =============================================================================

reg_landscape = (
    corr_wide
    .join(
        state_wide,
        how="outer"
    )
)


reg_landscape.insert(
    0,
    "Regulon",
    reg_landscape.index
)


reg_landscape[
    "TF"
] = (
    reg_landscape[
        "Regulon"
    ]
    .str.replace(
        r"\(\+\)$",
        "",
        regex=True
    )
)


# -------------------------------------------------------------------------
# Equal-weight mean correlation across the two patients
# -------------------------------------------------------------------------

for feature in feature_map.keys():

    cols = [
        f"rho_{feature}_HCC09",
        f"rho_{feature}_HCC10"
    ]

    reg_landscape[
        f"Mean_rho_{feature}"
    ] = (
        reg_landscape[
            cols
        ]
        .mean(
            axis=1
        )
    )


print(
    "Integrated landscape:",
    reg_landscape.shape
)

display(
    reg_landscape.head()
)

In [ ]:
# =============================================================================
# Step29H Cell 79
# Cross-patient mechanistic consistency criteria
# =============================================================================

def both_positive(feature):

    return (
        (
            reg_landscape[
                f"rho_{feature}_HCC09"
            ] > 0
        )
        &
        (
            reg_landscape[
                f"rho_{feature}_HCC10"
            ] > 0
        )
    )


def both_negative(feature):

    return (
        (
            reg_landscape[
                f"rho_{feature}_HCC09"
            ] < 0
        )
        &
        (
            reg_landscape[
                f"rho_{feature}_HCC10"
            ] < 0
        )
    )


# -------------------------------------------------------------------------
# MS3 / CIN side
# -------------------------------------------------------------------------

reg_landscape[
    "MS3_Crit_Delta"
] = (
    reg_landscape[
        "Delta_positive_both"
    ]
)

reg_landscape[
    "MS3_Crit_PT"
] = both_positive(
    "PT"
)

reg_landscape[
    "MS3_Crit_Core50"
] = both_positive(
    "Core50"
)

reg_landscape[
    "MS3_Crit_G2M"
] = both_positive(
    "G2M"
)

reg_landscape[
    "MS3_Crit_Mitotic"
] = both_positive(
    "Mitotic"
)

reg_landscape[
    "MS3_Crit_Spindle"
] = both_positive(
    "Spindle"
)

reg_landscape[
    "MS3_Crit_HepaticLoss"
] = both_negative(
    "Hepatic"
)

reg_landscape[
    "MS3_Crit_BileLoss"
] = both_negative(
    "BileAcid"
)


MS3_CRITERIA = [
    "MS3_Crit_Delta",
    "MS3_Crit_PT",
    "MS3_Crit_Core50",
    "MS3_Crit_G2M",
    "MS3_Crit_Mitotic",
    "MS3_Crit_Spindle",
    "MS3_Crit_HepaticLoss",
    "MS3_Crit_BileLoss"
]


reg_landscape[
    "MS3_consistency_score"
] = (
    reg_landscape[
        MS3_CRITERIA
    ]
    .fillna(False)
    .astype(int)
    .sum(
        axis=1
    )
)


# -------------------------------------------------------------------------
# MS6 / hepatic side = reciprocal pattern
# -------------------------------------------------------------------------

reg_landscape[
    "MS6_Crit_Delta"
] = (
    reg_landscape[
        "Delta_negative_both"
    ]
)

reg_landscape[
    "MS6_Crit_PT"
] = both_negative(
    "PT"
)

reg_landscape[
    "MS6_Crit_Core50"
] = both_negative(
    "Core50"
)

reg_landscape[
    "MS6_Crit_G2M"
] = both_negative(
    "G2M"
)

reg_landscape[
    "MS6_Crit_Mitotic"
] = both_negative(
    "Mitotic"
)

reg_landscape[
    "MS6_Crit_Spindle"
] = both_negative(
    "Spindle"
)

reg_landscape[
    "MS6_Crit_Hepatic"
] = both_positive(
    "Hepatic"
)

reg_landscape[
    "MS6_Crit_Bile"
] = both_positive(
    "BileAcid"
)


MS6_CRITERIA = [
    "MS6_Crit_Delta",
    "MS6_Crit_PT",
    "MS6_Crit_Core50",
    "MS6_Crit_G2M",
    "MS6_Crit_Mitotic",
    "MS6_Crit_Spindle",
    "MS6_Crit_Hepatic",
    "MS6_Crit_Bile"
]


reg_landscape[
    "MS6_consistency_score"
] = (
    reg_landscape[
        MS6_CRITERIA
    ]
    .fillna(False)
    .astype(int)
    .sum(
        axis=1
    )
)


print(
    "Consistency scores calculated."
)

In [ ]:
# =============================================================================
# Step29H Cell 80
# Provisional mechanistic shortlist
# =============================================================================

reg_landscape[
    "MS3_shortlist"
] = (
    (
        reg_landscape[
            "MS3_consistency_score"
        ] >= 6
    )
    &
    reg_landscape[
        "Delta_positive_both"
    ]
    &
    (
        reg_landscape[
            "Mean_Delta_MS3_MS6"
        ] >= 0.25
    )
)


reg_landscape[
    "MS6_shortlist"
] = (
    (
        reg_landscape[
            "MS6_consistency_score"
        ] >= 6
    )
    &
    reg_landscape[
        "Delta_negative_both"
    ]
    &
    (
        reg_landscape[
            "Mean_Delta_MS3_MS6"
        ] <= -0.25
    )
)


reg_landscape[
    "MS5_transition_shortlist"
] = (
    reg_landscape[
        "MS5_peak_both"
    ]
    &
    (
        reg_landscape[
            "Mean_MS5_peak_margin"
        ] >= 0.15
    )
)


print(
    "MS3 shortlist:",
    int(
        reg_landscape[
            "MS3_shortlist"
        ].sum()
    )
)

print(
    "MS6 shortlist:",
    int(
        reg_landscape[
            "MS6_shortlist"
        ].sum()
    )
)

print(
    "MS5-transition shortlist:",
    int(
        reg_landscape[
            "MS5_transition_shortlist"
        ].sum()
    )
)

In [ ]:
# =============================================================================
# Step29H Cell 81
# MS3 / CIN-associated regulon candidates
# =============================================================================

ms3_candidates = (
    reg_landscape.loc[
        reg_landscape[
            "MS3_shortlist"
        ]
    ]
    .sort_values(
        [
            "MS3_consistency_score",
            "Mean_Delta_MS3_MS6",
            "Mean_rho_Core50",
            "Mean_rho_Mitotic"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
)


ms3_display_cols = [
    "TF",
    "Regulon",

    "MS3_consistency_score",

    "Mean_Delta_MS3_MS6",

    "Mean_rho_PT",
    "Mean_rho_Core50",
    "Mean_rho_G2M",
    "Mean_rho_Mitotic",
    "Mean_rho_Spindle",

    "Mean_rho_Hepatic",
    "Mean_rho_BileAcid",

    "Delta_MS3_MS6_HCC09",
    "Delta_MS3_MS6_HCC10"
]


print(
    "MS3/CIN shortlist:",
    len(ms3_candidates)
)

display(
    ms3_candidates[
        ms3_display_cols
    ].head(30)
)

In [ ]:
# =============================================================================
# Step29H Cell 82
# MS6 / hepatic-differentiation regulon candidates
# =============================================================================

ms6_candidates = (
    reg_landscape.loc[
        reg_landscape[
            "MS6_shortlist"
        ]
    ]
    .sort_values(
        [
            "MS6_consistency_score",
            "Mean_Delta_MS3_MS6",
            "Mean_rho_Hepatic",
            "Mean_rho_BileAcid"
        ],
        ascending=[
            False,
            True,
            False,
            False
        ]
    )
)


ms6_display_cols = [
    "TF",
    "Regulon",

    "MS6_consistency_score",

    "Mean_Delta_MS3_MS6",

    "Mean_rho_PT",
    "Mean_rho_Core50",

    "Mean_rho_Hepatic",
    "Mean_rho_BileAcid",

    "Mean_rho_G2M",
    "Mean_rho_Mitotic",
    "Mean_rho_Spindle",

    "Delta_MS3_MS6_HCC09",
    "Delta_MS3_MS6_HCC10"
]


print(
    "MS6/Hepatic shortlist:",
    len(ms6_candidates)
)

display(
    ms6_candidates[
        ms6_display_cols
    ].head(30)
)

In [ ]:
# =============================================================================
# Step29H Cell 83
# MS5-centered transitional regulon candidates
# =============================================================================

ms5_candidates = (
    reg_landscape.loc[
        reg_landscape[
            "MS5_transition_shortlist"
        ]
    ]
    .sort_values(
        "Mean_MS5_peak_margin",
        ascending=False
    )
)


ms5_display_cols = [
    "TF",
    "Regulon",

    "Mean_MS5_peak_margin",

    "MS5_peak_margin_HCC09",
    "MS5_peak_margin_HCC10",

    "MS6_median_z_HCC09",
    "MS5_median_z_HCC09",
    "MS3_median_z_HCC09",

    "MS6_median_z_HCC10",
    "MS5_median_z_HCC10",
    "MS3_median_z_HCC10",

    "Mean_rho_PT",
    "Mean_rho_Core50"
]


print(
    "MS5-transition shortlist:",
    len(ms5_candidates)
)

display(
    ms5_candidates[
        ms5_display_cols
    ].head(30)
)

In [ ]:
# =============================================================================
# Step29H Cell 84
# Sentinel regulator final audit
# =============================================================================

sentinel_tfs = [
    "HNF4A",
    "HNF1A",

    "FOXM1",
    "E2F1",
    "E2F8",
    "MYBL2",
    "MYC",

    "KLF6",
    "CEBPA",
    "CEBPB",
    "XBP1",
    "ATF3",
    "JUNB",
    "JUND",
    "SOX4",
    "HMGA1"
]


sentinel_final = (
    reg_landscape.loc[
        reg_landscape[
            "TF"
        ].isin(
            sentinel_tfs
        )
    ]
    .copy()
)


sentinel_cols = [
    "TF",
    "Regulon",

    "MS3_consistency_score",
    "MS6_consistency_score",

    "Mean_Delta_MS3_MS6",
    "Mean_MS5_peak_margin",

    "Mean_rho_PT",
    "Mean_rho_Core50",

    "Mean_rho_G2M",
    "Mean_rho_Mitotic",
    "Mean_rho_Spindle",

    "Mean_rho_Hepatic",
    "Mean_rho_BileAcid",

    "MS3_shortlist",
    "MS6_shortlist",
    "MS5_transition_shortlist"
]


sentinel_final = (
    sentinel_final[
        sentinel_cols
    ]
    .sort_values(
        [
            "MS3_consistency_score",
            "MS6_consistency_score"
        ],
        ascending=False
    )
)


display(
    sentinel_final
)

In [ ]:
# =============================================================================
# Step29H Cell 85
# Save mechanistic validation tables
# =============================================================================

corr_long.to_csv(
    TABLE_DIR /
    "Step29H_PatientSpecific_Regulon_Correlations.csv",
    index=False
)

state_effect_long.to_csv(
    TABLE_DIR /
    "Step29H_PatientSpecific_StateEffects.csv",
    index=False
)

reg_landscape.to_csv(
    TABLE_DIR /
    "Step29H_Regulatory_Landscape.csv",
    index=False
)

ms3_candidates.to_csv(
    TABLE_DIR /
    "Step29H_MS3_CIN_Regulon_Shortlist.csv",
    index=False
)

ms6_candidates.to_csv(
    TABLE_DIR /
    "Step29H_MS6_Hepatic_Regulon_Shortlist.csv",
    index=False
)

ms5_candidates.to_csv(
    TABLE_DIR /
    "Step29H_MS5_Transition_Regulon_Shortlist.csv",
    index=False
)

sentinel_final.to_csv(
    TABLE_DIR /
    "Step29H_Sentinel_Regulator_FinalAudit.csv",
    index=False
)

print(
    "STEP29H tables saved."
)

In [ ]:
# =============================================================================
# Step29H Cell 86
# REPORT — regulatory architecture screening
# =============================================================================

STEP29H_PASS = (
    len(reg_landscape) == 278
    and
    len(ms3_candidates) >= 0
    and
    len(ms6_candidates) >= 0
    and
    len(ms5_candidates) >= 0
)


print("=" * 100)

print(
    "STEP29H REGULATORY ARCHITECTURE REPORT"
)

print("=" * 100)


print(
    "\nFrozen motif-supported regulons:",
    280
)

print(
    "Axis-informative regulons:",
    len(reg_landscape)
)

print(
    "Axis-nonvariable regulons excluded:",
    2
)


print(
    "\nCross-patient state-direction:"
)

print(
    "MS3 > MS6 in both patients:",
    int(
        reg_landscape[
            "Delta_positive_both"
        ].sum()
    )
)

print(
    "MS6 > MS3 in both patients:",
    int(
        reg_landscape[
            "Delta_negative_both"
        ].sum()
    )
)

print(
    "MS5 peak in both patients:",
    int(
        reg_landscape[
            "MS5_peak_both"
        ].sum()
    )
)


print(
    "\nProvisional mechanistic shortlists:"
)

print(
    "MS3/CIN:",
    len(ms3_candidates)
)

print(
    "MS6/Hepatic:",
    len(ms6_candidates)
)

print(
    "MS5 transition:",
    len(ms5_candidates)
)


print(
    "\nTOP MS3/CIN CANDIDATES:"
)

display(
    ms3_candidates[
        ms3_display_cols
    ].head(15)
)


print(
    "\nTOP MS6/HEPATIC CANDIDATES:"
)

display(
    ms6_candidates[
        ms6_display_cols
    ].head(15)
)


print(
    "\nTOP MS5-TRANSITION CANDIDATES:"
)

display(
    ms5_candidates[
        ms5_display_cols
    ].head(15)
)


print(
    "\nSENTINEL REGULATORS:"
)

display(
    sentinel_final
)


print(
    "\nSTEP29H_PASS:",
    STEP29H_PASS
)


print(
    "\nInterpretation boundary:"
)

print(
    "Regulon-state associations are cross-patient "
    "single-cell regulatory associations."
)

print(
    "They do not establish causal TF control or lineage conversion."
)

print(
    "Cell-level p-values were not used as the primary "
    "selection criterion."
)

print(
    "Final regulator locking requires biological review "
    "of the cross-patient effect architecture."
)

print(
    "\nSTEP29H SCREENING COMPLETE"
)

In [ ]:
# =============================================================================
# Step29 Final Lock
# Frozen regulatory architecture
# =============================================================================

STEP29_FINAL_LOCK = {
    "MS6_Hepatic_Identity": [
        "HNF4A",
        "NR1H4",
        "NR1I3",
        "CEBPA",
        "FOXA3"
    ],

    "MS5_Transitional_Plasticity": [
        "KLF4",
        "FOS",
        "FOSB",
        "ELF3",
        "SOX4"
    ],

    "MS3_Proliferative_CIN": [
        "FOXM1",
        "MYBL2",
        "E2F8",
        "TFDP1",
        "E2F1",
        "BRCA1"
    ]
}

import json

FINAL_LOCK_JSON = (
    AUDIT_DIR /
    "Step29_FINAL_RegulatoryArchitecture.json"
)

with open(
    FINAL_LOCK_JSON,
    "w"
) as f:

    json.dump(
        STEP29_FINAL_LOCK,
        f,
        indent=2
    )

print("=" * 100)
print("STEP29 FINAL LOCK")
print("=" * 100)

for layer, genes in STEP29_FINAL_LOCK.items():
    print(
        f"\n{layer}:"
    )
    print(
        ", ".join(genes)
    )

print(
    "\nFrozen output:"
)

print(
    FINAL_LOCK_JSON
)

print(
    "\nSTEP29 FINAL LOCK: COMPLETE"
)